# Gabriel et al. (2025) on dandiset 001887 and the paper's Zenodo record

**Paper.** Gabriel CJ, Gupta T, Sanchez-Fuentes A, Zeidler Z, Wilke SA, DeNardo LA (2025).
Transformations in prefrontal ensemble activity underlying rapid threat avoidance learning.
*Current Biology* 35, 1128-1136. [doi:10.1016/j.cub.2025.01.010](https://doi.org/10.1016/j.cub.2025.01.010).
The claims file was written from the NIH author manuscript in PubMed Central (PMC11916606) with its
STAR Methods; the supplement was not read by the reader.

**Dandiset.** [DANDI:001887](https://dandiarchive.org/dandiset/001887/0.260902.1724), version
0.260902.1724, the only published version (2026-09-02). The paper cites no dandiset version (it
predates the dandiset), so the most recent published version is pinned; the draft shows the same
56 assets and size.

**Zenodo record.** The dandiset holds no calcium imaging and BehaviorDEPOT labels for only two
sessions. The paper's STAR Methods name its "Analyzed datasets" on Zenodo
([doi:10.5281/zenodo.13381210](https://doi.org/10.5281/zenodo.13381210), Day 1.zip and Day 2.zip),
which hold for every imaged session the Minian traces, the BehaviorDEPOT output, the trial results and
the camera and miniscope timestamps. The claims are tested on that record (`data: elsewhere`), streamed
by HTTP range requests on the zip members, and counted apart from the dandiset. The analysis code read is
Zenodo doi:10.5281/zenodo.13351302.

| Claims | Status here |
|---|---|
| zenodo-dandiset-link (added by the author) | tested |
| data-neurons | tested |
| data-registered-cells, fig1o-tone-to-mixed, fig1o-nonmod-to-mixed | blocked: no registration is deposited; CellReg is MATLAB |
| fig1e, fig1f, fig1g, fig1h (behaviour) | tested |
| fig1kl-overall, fig1kl-day1-categories, disc-tone-cells-baseline, fig1k-mixed-day2 | tested |
| fig1p, fig1q, fig1pq, fig1r, figs1h (mean activity) | tested |
| figs2-innate-behaviors | blocked: no jumping or rearing labels; BehaviorDEPOT is MATLAB (the escape part is tested) |
| fig2-day1, fig2-day2, fig2e, fig2f | tested |
| fig3a (three claims), fig3b, fig3c (decoding) | tested |
| fig4b-hybrid-model-best | tested |
| figs3gh-behavior-similar | tested |
| fig4e to fig4l, figs3ab, figs3cd, figs3ef, figs4ab, figs4cd (CILDS, 13 claims) | blocked: CILDS is MATLAB |

The notebook takes about 41 minutes: 9 for streaming, 7 for the ROC runs and 25 for the 342 decoders of Figure 3
(four processes).

In [ ]:
import multiprocessing as mp
import os
import time
import warnings
import zlib
from concurrent.futures import ProcessPoolExecutor
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
import statsmodels.formula.api as smf
from scipy import stats

import pma

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

started = time.time()
meta = pma.dandi_metadata()
assets = pma.dandi_assets()
nwb = pma.read_nwb_all(assets)
print(f"dandiset {pma.DANDISET} version {pma.VERSION}: {len(assets)} assets, "
      f"{len(nwb)} NWB files read in {time.time() - started:.0f} s")
S = pma.zenodo_sessions()
beh = pma.load_parallel(pma.load_behaviour, S)
print(f"Zenodo record {pma.ZENODO_RECORD}: behaviour of {len(beh)} sessions read, "
      f"{time.time() - started:.0f} s")
ca = pma.load_parallel(pma.load_calcium, S)
print(f"calcium traces of {len(ca)} sessions read, {time.time() - started:.0f} s")
CA_T = {s: pma.ca_times(ca[s]) for s in S}
GROUPS = ("Shock", "NonShock")

## Data check

### The dandiset

In [ ]:
print("Dandiset title:", meta["name"])
print("Description:", meta.get("description"))
print("Assets summary:", {k: meta["assetsSummary"].get(k) for k in ("numberOfSubjects", "numberOfFiles", "numberOfBytes", "variableMeasured")})
kinds = pd.Series([a["path"].rsplit(".", 1)[-1] for a in assets]).value_counts()
print("Asset types:", kinds.to_dict())
rows = []
for k, r in sorted(nwb.items()):
    tr = r["trials"]
    rows.append(dict(subject=r["subject"], sex=r["sex"], session=r["session"], schema=r["notes"]["conversion_schema"],
                     source=r["notes"]["source"], start=r["start_time"][:19], kB=r["size"] // 1000,
                     trials=None if tr is None else len(tr["start_time"]),
                     shock_trials=None if tr is None else int(np.sum(tr["shock_present"])),
                     shock_intervals=None if r["shocks"] is None else len(r["shocks"]),
                     video_frames=r["video_frames"], behaviour_series=len(r["processing"]), imaging=r["imaging"]))
inv = pd.DataFrame(rows)
print(inv.to_string(index=False))
print("\nSubjects:", inv.subject.nunique(), "| sessions by name:", inv.session.value_counts().to_dict())
print("Files with any imaging data:", int(inv.imaging.sum()), "| files with BehaviorDEPOT arrays:",
      inv[inv.behaviour_series > 0][["subject", "session"]].values.tolist())

In [ ]:
# Recordings deposited more than once: identical trial tables or the same source file.
dup = inv[inv.source.duplicated(keep=False)]
print("Files converted from the same source file:\n", dup[["subject", "session", "source", "video_frames"]].to_string(index=False))
keys = [k for k, r in nwb.items() if r["trials"] is not None]
for a, b in combinations(keys, 2):
    ta, tb = nwb[a]["trials"]["start_time"], nwb[b]["trials"]["start_time"]
    if len(ta) == len(tb) and np.array_equal(ta, tb):
        print("identical trial start times:", a, b)
ns_shock = inv[(inv.session.str.contains("NoShock")) & (inv.shock_intervals.fillna(0) > 0)]
print("\nNon-shocked sessions that carry shock intervals:\n", ns_shock[["subject", "session", "shock_trials", "shock_intervals", "source"]].to_string(index=False))

The dandiset holds 30 NWB files for 16 mice (9 shocked, 7 non-shocked) and 26 behaviour videos as
external AVI files. No file holds calcium imaging, cell footprints or traces: the description
("Data collected for/analyzed in Gabriel et al (2025)") is not supported for the imaging. Two files
(F1 and F2, day 1) hold the BehaviorDEPOT arrays and no trial table, and their session start time is
2026-08-25, the conversion date. F6 day 1 holds only the video. The other 26 hold a trial table (tone
and shock times) and no behaviour labels. The F7 day 2 file (non-shocked) was converted from M7's day 2
timestamp file: its trial table is identical to M7's and lists 12 shock trials. The F3 day 1 file
(non-shocked) lists 9 shock trials. The F3 day 2 table has 9 trials, the first at 0 s. None of these
files is used for a result below except as the link of the next section.

### The Zenodo record

In [ ]:
zrows = []
for s in S:
    b, c = beh[s], ca[s]
    cam_fps = (len(b["cam_t"]) - 1) / (b["cam_t"][-1] - b["cam_t"][0])
    ca_t = CA_T[s]
    tw = pma.tone_windows(b, "clock")
    r = nwb.get(pma.nwb_key(s))
    tr = None if r is None else r["trials"]
    on = tw[:, 0]
    off = None
    if tr is not None and len(tr["start_time"]) == len(on):
        d = (b["tone_bouts"][:, 0] / pma.CODE_FPS) - tr["start_time"]
        off = (np.nanmin(d), np.nanmax(d))
    zrows.append(dict(day=s[0], group=s[1], mouse=s[2], cells=c["dff"].shape[0], ca_frames=c["dff"].shape[1],
                      ca_fps=round((len(ca_t) - 1) / (ca_t[-1] - ca_t[0]), 2), ca_min=round((ca_t[-1] - ca_t[0]) / 60, 1),
                      cam_frames=len(b["platform"]), cam_ts=len(b["cam_t"]), cam_fps=round(cam_fps, 2),
                      tones=int(np.sum(~np.isnan(on))), bout_frames=int(np.nanmedian(np.diff(b["tone_bouts"], axis=1))),
                      dff_min=round(float(c["dff"].min()), 2), dff_max=round(float(c["dff"].max()), 1),
                      plat_frac=round(b["platform"].mean(), 2), frz_frac=round(b["freezing"].mean(), 2),
                      nwb_trials=None if tr is None else len(tr["start_time"]),
                      offset_s=None if off is None else f"{off[0]:.2f} to {off[1]:.2f}",
                      hand_scored=b["pma"].get("avoids") is not None))
Z = pd.DataFrame(zrows)
print(Z.to_string(index=False))
print("\nSessions per day and group:\n", Z.groupby(["day", "group"]).mouse.apply(lambda x: f"{len(x)}: {', '.join(x)}").to_string())
print("\nCells per day and group:\n", Z.groupby(["day", "group"]).cells.sum().to_string())

In [ ]:
# What the Zenodo files hold: every field of Behavior.mat and every variable of the minian file.
from collections import Counter
fc = Counter(f for s in S for f in beh[s]["fields"])
print(f"Behavior.mat fields (sessions holding each, of {len(S)}):")
print("  " + ", ".join(f"{k} ({v})" for k, v in sorted(fc.items())))
print("Fields naming jumping, rearing or escape:", [f for f in fc if any(w in f.lower() for w in ("jump", "rear", "escape"))])
vc = Counter(n for s in S for n, _ in ca[s]["variables"])
print(f"\nminian_processed.mat variables (sessions holding each, of {len(S)}):", dict(sorted(vc.items())))
s0 = S[0]
print(f"shapes in {s0[2]} day {s0[0]}:", {n: tuple(int(x) for x in sh) for n, sh in ca[s0]["variables"]})
print("Sessions whose ta_rois (footprints) count equals the ta_dff cells:",
      sum(dict(ca[s]["variables"]).get("ta_rois", (0, 0, -1))[-1] == ca[s]["dff"].shape[0] for s in S), "of", len(S))

The Zenodo record holds 30 sessions: day 1, 9 shocked and 7 non-shocked mice; day 2, 7 and 7 (F1 and
F2 have no day 2). The paper's Figure 1 behaviour legend gives 11 and 9 mice on day 1 and 8 and 8 on day
2: the behaviour-only mice are not deposited. Calcium frames run at 14.9 frames/s (every second
miniscope timestamp, as the 2x temporal downsampling implies) for 28 to 33 minutes, and the miniscope
timestamps number twice the calcium frames in every session. Every session has 12 tones except F3 on
day 2 (10; tones 2 and 3 are NaN in BehaviorDEPOT). The BehaviorDEPOT tone bouts last 751 frames, 30 s at
25 frames/s. The camera timestamps give 25.0 frames/s in 24 sessions but 28.2 to 29.8 frames/s in six
(F5 and M5 on both days, F6 on both days): there the bouts, placed at 25 frames/s, drift from the real
tones by up to several minutes by the end of the session. This is a finding about the paper's
processing; the `clock` tone windows (pma.tone_windows) correct it, and the `paper` windows keep it.

What the files hold (the cell after the table): Behavior.mat has the platform, freezing, tone,
platform entry and exit, tone onset and offset and inter-platform-interval fields in all 30 sessions,
and no field for jumping, rearing or escape. Each minian file holds `ta_dff`, `ta_sigfn` and the cell
footprints `ta_rois` (500 by 300 pixels per cell, one per `ta_dff` cell in all 30 sessions), besides
unrefined versions of each. So the footprints a registration would need are deposited; the
registration itself is not.

### Clock check

Two contrasts test where the tones and shocks were. First, behaviour: the hand-scored trial
outcomes against the code's rule on each tone reading. Second, calcium: in shocked mice the
population response in the 2 s after the shock onset, against the 2 s before, under each tone
reading and with the shock times moved by 60 s.

In [ ]:
crow = []
for s in S:
    b = beh[s]
    man = b["pma"].get("avoids")
    if man is None:
        continue
    out = dict(day=s[0], group=s[1], mouse=s[2], cam_fps=Z.set_index(["day", "group", "mouse"]).loc[s[:3], "cam_fps"])
    for rd in ("paper", "clock"):
        m = pma.trial_measures_code(b) if rd == "paper" else pma.trial_measures_clock(b)
        ok = ~np.isnan(m["avoid"])
        out[f"rule_vs_hand_{rd}"] = int((m["avoid"][ok] != man).sum())
    crow.append(out)
CK = pd.DataFrame(crow)
CK["fast_camera"] = CK.cam_fps > 26
print("Trials where the code's success rule disagrees with the hand-scored outcome (12 trials per session):")
print(CK.to_string(index=False))
print("\nTotal disagreements by camera rate:\n", CK.groupby("fast_camera")[["rule_vs_hand_paper", "rule_vs_hand_clock"]].sum().to_string())

# The coding of the hand-scored outcome: its values against the rule and against the time on the
# platform in the first second of the shock window (clock windows), over all hand-scored trials.
hs = []
for s in S:
    man = beh[s]["pma"].get("avoids")
    if man is None:
        continue
    b = beh[s]
    m = pma.trial_measures_clock(b)
    ok = ~np.isnan(m["avoid"])
    shock_win = pma.epoch_fraction(b, "clock", pma.TONE_S - 2, pma.TONE_S - 1, "platform")[ok]
    hs += [dict(group=s[1], avoids=int(a), rule=int(r), plat_shock_s1=w) for a, r, w in zip(man, m["avoid"][ok], shock_win)]
HS = pd.DataFrame(hs)
print("\npma_results avoids against the code's rule (clock windows), trials:")
print(pd.crosstab(HS.avoids, HS.rule).to_string())
print("Fraction of the first shock second on the platform by avoids value:")
print(HS.groupby("avoids").plat_shock_s1.describe()[["count", "mean", "25%", "50%", "75%"]].round(3).to_string())

def shock_response(s, reading, shift=0.0):
    tw = pma.tone_windows(beh[s], reading)
    t = CA_T[s]
    x = ca[s]["dff"]
    z = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-9)
    pop = z.mean(0)
    d = []
    for off in tw[~np.isnan(tw[:, 0]), 1]:
        sh = off - 2.0 + shift
        pre, post = (t >= sh - 2) & (t < sh), (t >= sh) & (t < sh + 2)
        if pre.any() and post.any():
            d.append(pop[post].mean() - pop[pre].mean())
    return np.mean(d)

rows = []
for s in S:
    if s[1] != "Shock":
        continue
    rows.append(dict(day=s[0], mouse=s[2], cam_fps=Z.set_index(["day", "group", "mouse"]).loc[s[:3], "cam_fps"],
                     clock=shock_response(s, "clock"), paper=shock_response(s, "paper"),
                     clock_shift_60s=shock_response(s, "clock", 60.0), clock_shift_m60s=shock_response(s, "clock", -60.0)))
SR = pd.DataFrame(rows)
print("\nPopulation z-scored DeltaF/F, mean over cells, 2 s after minus 2 s before the shock onset, mean over trials:")
print(SR.round(3).to_string(index=False))

In [ ]:
# Do the calcium frames and the camera labels share a clock? Per session, the correlation of every
# cell's DeltaF/F with the freezing labels (and the platform labels) shifted by -10 to +10 s; the
# mean absolute correlation over cells peaks at the lag by which calcium follows the labels.
def label_lag(s, which, max_s=10.0):
    t, x = CA_T[s], ca[s]["dff"]
    n = x.shape[1]
    z = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-12)
    v = pma.to_ca(beh[s], t, which).astype(float)
    v = (v - v.mean()) / v.std()
    r = np.fft.irfft(np.fft.rfft(z, axis=1) * np.conj(np.fft.rfft(v)), n=n, axis=1) / n   # r[c, k] = corr(x(t), v(t - k))
    k = int(round(max_s * pma.CA_FPS))
    lags = np.r_[np.arange(-k, 0) + n, np.arange(0, k + 1)]
    m = np.abs(r[:, lags]).mean(0)
    lag_s = np.r_[np.arange(-k, 0), np.arange(0, k + 1)] / ((len(t) - 1) / (t[-1] - t[0]))
    return lag_s[np.argmax(m)], m.max(), m[k]

lrow = []
for s in S:
    out = dict(day=s[0], group=s[1], mouse=s[2], cam_fps=Z.set_index(["day", "group", "mouse"]).loc[s[:3], "cam_fps"])
    for which in ("freezing", "platform"):
        lg, pk, at0 = label_lag(s, which)
        out[f"{which}_peak_lag_s"], out[f"{which}_peak_r"], out[f"{which}_r_at_0"] = round(lg, 2), round(pk, 3), round(at0, 3)
    lrow.append(out)
LAG = pd.DataFrame(lrow)
print(LAG.to_string(index=False))
print("\nFreezing peak lag (s), by camera rate:", LAG.groupby(LAG.cam_fps > 26).freezing_peak_lag_s.describe()[["count", "mean", "min", "max"]].round(2).to_dict("index"))

For the six fast-camera sessions the code's success rule disagrees with the hand-scored outcomes on
15 trials with the paper's tone bouts and on 4 with the `clock` windows; for the 25 frames/s sessions
the readings coincide (21 and 20 disagreements, the rule's own error against hand scoring). So the
behaviour labels and the `clock` tone times share one clock, and the paper's bouts do not in the
fast-camera sessions. The hand-scored `avoids` is 1 for a success: over the 252 hand-scored trials it
agrees with the rule on 228 (147 both 1, 81 both 0), and trials scored 1 spent 0.92 of the first
shock second on the platform on average, against 0.13 for trials scored 0.

The shock contrast does not decide the calcium clock: the population response to the shock is
within plus or minus 0.08 z under every reading, including the 60 s shifts. The label-lag check
does: in all 30 sessions the mean absolute correlation of the cells with the freezing labels peaks
when calcium follows the labels by 0.9 to 3.8 s (mean 2.0 s in the 25 frames/s sessions, 1.7 s in the
fast-camera ones), with no session offset by more than that, the delay expected from GCaMP7f and
a freezing label that starts after 0.9 s of stillness. The platform labels peak within -1.4 to
+10 s, a flatter curve (peak correlation within 0.02 of the value at lag 0). So calcium and camera
labels are consistent with one clock in every session, to within a few seconds. All analyses
below use the `clock` windows and show the paper's windows beside them.

## Link between the dandiset and the Zenodo record

In [ ]:
lrows = []
for s in S:
    r = nwb.get(pma.nwb_key(s))
    b = beh[s]
    row = dict(day=s[0], group=s[1], mouse=s[2], in_dandiset=r is not None)
    if r is not None:
        tr = r["trials"]
        row["dandiset_trials"] = None if tr is None else len(tr["start_time"])
        row["zenodo_tones"] = int(np.sum(~np.isnan(b["tone_bouts"][:, 0])))
        if tr is not None and len(tr["start_time"]) == len(b["tone_bouts"]):
            d = b["tone_bouts"][:, 0] / pma.CODE_FPS - tr["start_time"]
            row["onset_offset_spread_s"] = round(float(np.nanmax(d) - np.nanmin(d)), 3)
        else:
            row["onset_offset_spread_s"] = None
        row["video_frames"] = r["video_frames"]
        row["zenodo_cam_frames"] = len(b["platform"])
        if r.get("beh") is not None:
            pv = r["beh"]["Behavior_Platform_Vector"].astype(bool)
            fv = r["beh"]["Behavior_Freezing_Vector"].astype(bool)
            row["dandiset_labels_equal"] = bool(len(pv) == len(b["platform"]) and np.array_equal(pv, b["platform"]) and np.array_equal(fv, b["freezing"]))
    lrows.append(row)
L = pd.DataFrame(lrows)
print(L.to_string(index=False))
matched = L.in_dandiset & (L.onset_offset_spread_s.fillna(99) < 0.1)
print(f"\nZenodo sessions with a dandiset file: {int(L.in_dandiset.sum())} of {len(L)}")
print(f"... whose dandiset trial table gives the Zenodo tone onsets up to one offset (spread < 0.1 s): {int(matched.sum())}")
print("Sessions without such a table:", L[~matched][["day", "group", "mouse"]].values.tolist())
print("Dandiset files with no Zenodo session:", sorted(set(nwb) - {pma.nwb_key(s) for s in S}))

**Result: partially reproduced.** All 30 Zenodo sessions have a dandiset file with the same mouse
and day, and 25 of them have a trial table that gives the Zenodo tone onsets up to one offset per
session. The other five are F1 and F2 day 1 (no trial table), F6 day 1 (video only), F3 day 2 (9 trials
against 10 tones) and F7 day 2 (M7's table). The BehaviorDEPOT arrays of F1 and F2 in the dandiset
equal the Zenodo ones.

## Cell counts

In [ ]:
cc = Z.groupby(["day", "group"]).agg(mice=("mouse", "count"), cells=("cells", "sum")).reset_index()
print(cc.to_string(index=False))
print("Expected: day 1 Shock 1394 cells from 9 mice, NonShock 847 from 7; day 2 Shock 1031, NonShock 967.")

**Result: reproduced.** The Zenodo traces (`ta_dff`, the traces the paper's code uses) hold 1394
cells from 9 shocked mice and 847 from 7 non-shocked mice on day 1, and 1031 and 967 on day 2, the
paper's counts exactly. The dandiset holds none of them.

In [ ]:
names = sorted({n.rsplit("/", 1)[-1] for d in (1, 2) for n in pma.zenodo_listing(d) if not n.endswith("/")})
generic = sorted({("<mouse>_" + n.split("_", 2)[-1]) if n[:1] in "FM" and "_T" in n[:5] else n for n in names})
print("File names in the Zenodo record (mouse prefix replaced):")
for n in generic:
    print("  ", n)
print("Any name mentioning registration or CellReg:", [n for n in names if "reg" in n.lower()])
print("Any registration series in the dandiset:", [k for r in nwb.values() for k in r["processing"] if "reg" in k.lower()])

**Result: blocked (not runnable).** Neither the Zenodo record nor the dandiset holds a cell
registration across days (no CellReg output, no file naming a registration), so the 463 and 322
registered cells cannot be counted, and the two Figure 1O claims, which use them, are blocked too.
The footprints it would start from are deposited (`ta_rois`, data check), but the paper's
registration is CellReg, which is MATLAB, and the criterion's zero tolerance needs that method.

## Behaviour (Figure 1E to 1H)

In [ ]:
TT = {rd: pma.trial_table(S, beh, rd) for rd in ("clock", "paper")}

def block_stats(T, measure, rising=True):
    B = T.groupby(["day", "group", "mouse", "block"])[measure].mean().reset_index()
    res = {}
    for day in (1, 2):
        d = B[B.day == day].dropna()
        full = d.groupby("mouse").block.nunique()
        d = d[d.mouse.isin(full[full == 4].index)]
        aov = pg.mixed_anova(d, dv=measure, within="block", between="group", subject="mouse", correction=True)
        sh = d[d.group == "Shock"].pivot(index="mouse", columns="block", values=measure)
        ns = d[d.group == "NonShock"].pivot(index="mouse", columns="block", values=measure)
        within = {pr: stats.ttest_rel(sh[pr[1]], sh[pr[0]]).pvalue for pr in combinations(range(1, 5), 2)}
        between = {k: stats.ttest_ind(sh[k], ns[k]).pvalue for k in range(1, 5)}
        res[day] = dict(means=d.groupby(["group", "block"])[measure].mean().unstack(), n=d.groupby("group").mouse.nunique().to_dict(),
                        aov=aov, within_all={k: pma.sidak(p, 6) for k, p in within.items()},
                        within_b1={k: pma.sidak(p, 3) for k, p in within.items() if k[0] == 1},
                        between={k: pma.sidak(p, 4) for k, p in between.items()})
    m = res[1]["means"]
    direction = m.loc["Shock", 4] > m.loc["Shock", 1] if rising else m.loc["Shock", 4] < m.loc["Shock", 1]
    verdict = {}
    for fam in ("within_all", "within_b1"):
        w = min(res[1][fam].values()) < 0.05
        btw = min(min(res[d]["between"].values()) for d in (1, 2)) < 0.05
        verdict[fam] = bool(direction and w and btw)
    return res, bool(direction), verdict

def show_behaviour(measure, rising=True):
    out = {}
    for rd in ("clock", "paper"):
        for succ in ("success",):
            res, direction, verdict = block_stats(TT[rd], measure, rising)
            print(f"=== {measure}, tone reading {rd}")
            for day in (1, 2):
                r = res[day]
                print(f"day {day}, mice {r['n']}; block means:")
                print(r["means"].round(3).to_string())
                a = r["aov"].set_index("Source")
                print("  mixed ANOVA: " + "; ".join(f"{src} F={a.loc[src, 'F']:.2f} p={a.loc[src, 'p_unc']:.3g}" + (f" (GG p={a.loc[src, 'p_GG_corr']:.3g})" if 'p_GG_corr' in a and not np.isnan(a.loc[src].get('p_GG_corr', np.nan)) else "") for src in a.index))
                print("  Shock blocks, Sidak over 6 pairs:", {f"{k[0]}-{k[1]}": round(v, 4) for k, v in r["within_all"].items()})
                print("  Shock blocks against block 1, Sidak over 3:", {f"{k[0]}-{k[1]}": round(v, 4) for k, v in r["within_b1"].items()})
                print("  Shock against NonShock per block, Sidak over 4:", {k: round(v, 4) for k, v in r["between"].items()})
            print(f"  day 1 Shock block 4 {'above' if rising else 'below'} block 1: {direction}; criterion met (all pairs / against block 1): {verdict}")
            out[rd] = verdict
    return out

In [ ]:
V_E = show_behaviour("platform")

**Result: reproduced.** On day 1 the shocked mice's fraction of the tone on the platform rises from
0.38 in block 1 to 0.78 in block 4 (`clock`), block 1 against 4 Sidak p = 0.0047 (all pairs) and 0.0024
(against block 1); shocked mice differ from non-shocked mice at day 1 block 4 (p = 0.048) and at day 2
blocks 2 to 4 (p = 0.0046 to 0.017). With the paper's tone bouts the day 1 rise is 0.38 to 0.74 (p =
0.0079 and 0.004) and day 2 blocks 2 to 4 differ (p = 0.0023 to 0.026). Group sizes are 9 and 7 mice on
day 1 and 7 and 7 on day 2, where the paper had 11, 9, 8 and 8.

In [ ]:
V_F = show_behaviour("success")
for rd in ("clock", "paper"):
    T = TT[rd]
    print(f"{rd}: hand-scored sessions {T[T.hand_scored].groupby(['day','group','mouse']).ngroups}; "
          f"rule against hand-scored outcome disagree on {int((T.success != T.success_rule).sum())} of {int(T.success.notna().sum())} trials")

**Result: reproduced.** The proportion of successful trials of shocked mice rises from 0.37 to 0.89
over day 1 (block 1 against 4 Sidak p = 0.0012 over all pairs, 0.0006 against block 1), and shocked mice
are above non-shocked mice at day 2 blocks 2 to 4 (p = 0.0014 to 0.032) under both tone readings.
Success is the hand-scored outcome in the 21 sessions that have one, else the code's rule.

In [ ]:
V_G = show_behaviour("freezing")

**Result: blocked (methods underspecified).** Freezing of shocked mice rises over day 1 from 0.22 to
0.42 (`clock`). Against block 1 the block 4 difference is Sidak p = 0.047; over all six pairs no
difference reaches p < 0.05 (smallest 0.092). With the paper's tone bouts both families pass (block 1
against 4, p = 0.0031 and 0.0015). Shocked mice freeze more than non-shocked mice on day 2 (block 2 p
= 0.026 and block 4 p = 0.0012, `clock`). The paper does not state the Sidak family, and on the
`clock` windows, the reading the data support, the two families the criterion names disagree.

In [ ]:
V_H = show_behaviour("latency", rising=False)

**Result: blocked (methods underspecified).** With the `clock` windows the shocked mice's latency
falls from 10.8 s in day 1 block 1 to 8.1 s in block 4; the only Sidak-significant block difference
is block 2 against 4 (p = 0.032, all pairs), none against block 1 (smallest 0.14), and the groups
differ at day 2 block 4 (p = 0.043). With the paper's tone bouts no within-group difference reaches p
< 0.05 (smallest 0.33 against block 1, 0.51 over all pairs). The criterion holds only on the `clock`
windows with all pairs; the Sidak family is not stated, and the two families disagree.

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(15, 6), sharex=True)
for i, meas in enumerate(("platform", "success", "freezing", "latency")):
    B = TT["clock"].groupby(["day", "group", "mouse", "block"])[meas].mean().reset_index()
    for j, day in enumerate((1, 2)):
        ax = axs[j, i]
        for g, col in zip(GROUPS, ("tab:red", "tab:gray")):
            d = B[(B.day == day) & (B.group == g)].groupby("block")[meas]
            ax.errorbar(d.mean().index, d.mean(), d.sem(), color=col, label=g, marker="o")
        ax.set_title(f"{meas}, day {day}")
axs[0, 0].legend()
plt.tight_layout()
plt.show()
for meas in ("platform", "success", "freezing", "latency"):
    v = TT["clock"][meas]
    print(f"{meas}: per-trial values range {v.min():.2f} to {v.max():.2f}; block means plotted, all within the axes")

## ROC encoding (Figure 1I to 1S)

The ROC is run on the `clock` tone windows (the data check) and, as a variant, on the paper's tone
bouts, with the calls for each period under each period reading (P1 tones 9 to 12, P2 tones 10 to 12)
and frame reading (F1, F2) of the claims file, and on the whole session for the mean-activity claims
(the code's session-wide vectors). Every claim below is shown under both tone readings.

In [ ]:
def periods(day):
    if day == 1:
        p = {"All": None, "BL-F1": ((1, 2, 3), "F1"), "BL-F2": ((1, 2, 3), "F2")}
        for pr, tones in (("P1", (9, 10, 11, 12)), ("P2", (10, 11, 12))):
            for fr in ("F1", "F2"):
                p[f"D1-{pr}-{fr}"] = (tones, fr)
        return p
    p = {"All": None}
    for pr, tones in (("P1", (9, 10, 11, 12)), ("P2", (10, 11, 12))):
        for fr in ("F1", "F2"):
            p[f"D2-{pr}-{fr}"] = (tones, fr)
    return p

FEATS = pma.CORE + ["tonePlatentVector", "itiPlatentVector", "tonePlatextVector", "itiPlatextVector"]

def run_roc(reading, which=None, tag=""):
    out, fe = {}, {}
    for s in S:
        cat = CA_T[s]
        F, ev, tw = pma.features(beh[s], cat, reading)
        fe[s] = (F, ev, tw)
        for pn, pdef in periods(s[0]).items():
            if which is not None and pn not in which:
                continue
            m = np.ones(len(cat), bool) if pdef is None else pma.period_mask(cat, tw, *pdef)
            rng = np.random.default_rng(zlib.crc32(f"{reading}{s[3]}{pn}{tag}".encode()))
            feats = FEATS if pn == "All" else pma.CORE
            A = np.full((len(ca[s]["dff"]), len(feats)), np.nan)
            C = np.zeros(A.shape, int)
            x = ca[s]["dff"][:, m]
            R = pma.roc_prepare(x)
            for j, fn in enumerate(feats):
                A[:, j], C[:, j] = pma.roc_calls(x, F[fn][m], rng, ranks=R)
            out[(s, pn)] = (feats, A, C)
    return out, fe

t0 = time.time()
ROC, FE = run_roc("clock")
ROCp, FEp = run_roc("paper")
TONE = ("clock", "paper")
ROCS = {"clock": (ROC, FE), "paper": (ROCp, FEp)}
print(f"ROC on {len(ROC)} session periods under each tone reading in {time.time() - t0:.0f} s")
for rd in TONE:
    cnt = []
    for day in (1, 2):
        for g in GROUPS:
            calls = np.vstack([ROCS[rd][0][(s, "All")][2] for s in S if s[0] == day and s[1] == g])
            cnt.append({"day": day, "group": g, **{f"{n[:-6]} +/-": f"{(calls[:, j] == 1).sum()}/{(calls[:, j] == -1).sum()}" for j, n in enumerate(FEATS)}})
    print(f"Whole-session calls, excited/suppressed cells, tone reading {rd}:")
    print(pd.DataFrame(cnt).to_string(index=False))
print("Paper (STAR Methods, Cells Analyzed for Figure 1), excited/suppressed: day 1 ToneOn Shk 61/52 NS 24/25, Frz Shk 162/334 NS 137/187, "
      "PlatEnt Shk 150/87 NS 95/66, PlatExt Shk 131/78 NS 66/67; day 2 ToneOn Shk 34/37 NS 34/22, Frz Shk 171/283 NS 150/198, "
      "PlatEnt Shk 112/80 NS 102/92, PlatExt Shk 104/74 NS 100/82")
ev_n = pd.DataFrame([dict(day=s[0], group=s[1], mouse=s[2], **{k: len(v) for k, v in FE[s][1].items()}) for s in S])
print("\nEvents per session (clock reading):")
print(ev_n.to_string(index=False))

In [ ]:
def cats(day, g, pn, roc=None):
    roc = ROC if roc is None else roc
    return np.vstack([pma.categories(roc[(s, pn)][2][:, :7]) for s in S if s[0] == day and s[1] == g])

def summary(K):
    n = (K != 0).sum(1)
    return dict(cells=len(K), responsive=int((n > 0).sum()), tone_only=int(((K[:, 0] != 0) & (n == 1)).sum()),
                platform_only=int(((K[:, 1] != 0) & (n == 1)).sum()), freezing_only=int(((K[:, 2] != 0) & (n == 1)).sum()),
                mixed=int((n >= 2).sum()), tone_any=int((K[:, 0] != 0).sum()))

READINGS = [(pr, fr) for pr in ("P1", "P2") for fr in ("F1", "F2")]
def pname(per, pr, fr):
    return f"BL-{fr}" if per == "BL" else f"{per}-{pr}-{fr}"
PERDAY = {"BL": 1, "D1": 1, "D2": 2}
SUMS = {rd: {} for rd in TONE}
for rd in TONE:
    for pr, fr in READINGS:
        for per, day in PERDAY.items():
            for g in GROUPS:
                SUMS[rd][(pr, fr, per, g)] = summary(cats(day, g, pname(per, pr, fr), ROCS[rd][0]))
SUM = SUMS["clock"]
SUMT = pd.DataFrame([dict(tone=rd, reading=f"{k[0]}-{k[1]}", period=k[2], group=k[3], **v) for rd in TONE for k, v in SUMS[rd].items()])
print(SUMT.to_string(index=False))

In [ ]:
# Where the event-call counts differ from the paper's: the same whole-session ROC for tone onsets,
# platform entries and exits under other event windows, platform bouts cleaned of brief flickers
# (gaps and bouts under 1 s), the paper's tone bouts, and three other shuffle seeds of the base run.
def clean_bouts(v, fps, gap_s=1.0, min_s=1.0):
    v = v.copy()
    for a, e in pma.bouts(~v):
        if a > 0 and e < len(v) - 1 and (e - a + 1) < gap_s * fps:
            v[a:e + 1] = True
    for a, e in pma.bouts(v):
        if (e - a + 1) < min_s * fps:
            v[a:e + 1] = False
    return v

PAPER_CALLS = {(1, "ToneOn"): "61/52, 24/25", (1, "PlatEnt"): "150/87, 95/66", (1, "PlatExt"): "131/78, 66/67",
               (2, "ToneOn"): "34/37, 34/22", (2, "PlatEnt"): "112/80, 102/92", (2, "PlatExt"): "104/74, 100/82"}
cnt = {}
for s in S:
    t, x, b = CA_T[s], ca[s]["dff"], beh[s]
    R = pma.roc_prepare(x)
    tw = pma.tone_windows(b, "clock")
    tw = tw[~np.isnan(tw[:, 0])]
    twp = pma.tone_windows(b, "paper")
    twp = twp[~np.isnan(twp[:, 0])]
    V = {("ToneOn", "0 to 5 s (base)"): pma.window_vector(t, tw[:, 0], 0, 5),
         ("ToneOn", "-1 to 4 s"): pma.window_vector(t, tw[:, 0], -1, 4),
         ("ToneOn", "2 to 7 s"): pma.window_vector(t, tw[:, 0], 2, 7),
         ("ToneOn", "paper tone bouts"): pma.window_vector(t, twp[:, 0], 0, 5)}
    cam_fps = (len(b["cam_t"]) - 1) / (b["cam_t"][-1] - b["cam_t"][0])
    bc = dict(b, platform=clean_bouts(b["platform"][:len(b["cam_t"])], cam_fps))
    for lab, bb in (("", b), ("cleaned bouts, ", bc)):
        pb = pma.bouts(pma.to_ca(bb, t, "platform"))
        for ev, col in (("PlatEnt", 0), ("PlatExt", 1)):
            e = t[pb[:, col]]
            V[(ev, lab + "-1 to 4 s" + (" (base)" if not lab else ""))] = pma.window_vector(t, e, -1, 4)
            if not lab:
                V[(ev, "0 to 4 s")] = pma.window_vector(t, e, 0, 4)
                V[(ev, "1 to 6 s")] = pma.window_vector(t, e, 1, 6)
    for k, v in V.items():
        seeds = range(4) if k[1].endswith("(base)") else (0,)
        for sd in seeds:
            rng = np.random.default_rng(zlib.crc32(f"events{sd}{s[3]}{k}".encode()))
            _, c = pma.roc_calls(x, v, rng, ranks=R)
            key = (s[0], k[0], k[1] + (f", seed {sd}" if len(seeds) > 1 else ""), s[1])
            e0, s0 = cnt.get(key, (0, 0))
            cnt[key] = (e0 + int((c == 1).sum()), s0 + int((c == -1).sum()))
EV = pd.DataFrame([dict(day=d, event=e, variant=v, group=g, calls=f"{a}/{b}") for (d, e, v, g), (a, b) in cnt.items()])
EV = EV.pivot_table(index=["day", "event", "variant"], columns="group", values="calls", aggfunc="first").reset_index()
EV["paper Shk, NS"] = [PAPER_CALLS[(d, e)] for d, e in zip(EV.day, EV.event)]
print("Excited/suppressed cells, whole session:")
print(EV[["day", "event", "variant", "Shock", "NonShock", "paper Shk, NS"]].to_string(index=False))

In [ ]:
rows = []
for rd in TONE:
    for pr, fr in READINGS:
        for per in PERDAY:
            a, b = SUMS[rd][(pr, fr, per, "Shock")], SUMS[rd][(pr, fr, per, "NonShock")]
            c, p = pma.chi2_yates(a["responsive"], a["cells"], b["responsive"], b["cells"])
            rows.append(dict(tone=rd, reading=f"{pr}-{fr}", period=per, shock=a["responsive"] / a["cells"], nonshock=b["responsive"] / b["cells"],
                             diff_pts=100 * (a["responsive"] / a["cells"] - b["responsive"] / b["cells"]), chi2=c, p=p))
OV = pd.DataFrame(rows)
print(OV.round(4).to_string(index=False))
for (rd, rg), d in OV.groupby(["tone", "reading"], sort=False):
    sim = (d.p >= 0.05).all()
    inc = all(d.set_index("period").loc["D2", g] > d.set_index("period").loc["BL", g] for g in ("shock", "nonshock"))
    print(f"{rd} {rg}: similar in all periods {sim}; D2 above BL in both groups {inc}; both {sim and inc}")

**Result: blocked (methods underspecified).** The fraction of responsive cells rises from BL to D2 in
both groups under every reading (shocked 0.46 to 0.52, non-shocked 0.47 to 0.55 under clock P1-F1).
Similarity holds in all three periods only under clock P2-F2 (smallest p = 0.14); under the other
clock readings the D1 fractions differ (non-shocked higher by 5.0 to 7.0 points, p = 0.0015 to
0.025), and with the paper's tone bouts D1 differs under every reading (p = 0.0003 to 0.047). The
claims file makes a claim whose readings disagree methods_underspecified.

In [ ]:
rows = []
for rd in TONE:
    for pr, fr in READINGS:
        for per in ("BL", "D1"):
            if per == "BL" and pr == "P2":
                continue
            a, b = SUMS[rd][(pr, fr, per, "Shock")], SUMS[rd][(pr, fr, per, "NonShock")]
            for k in ("tone_only", "platform_only", "freezing_only", "mixed"):
                c, p = pma.chi2_yates(a[k], a["responsive"], b[k], b["responsive"])
                rows.append(dict(tone=rd, reading=f"{pr}-{fr}" if per == "D1" else fr, period=per, category=k, shock=a[k] / a["responsive"],
                                 nonshock=b[k] / b["responsive"], diff_pts=100 * (a[k] / a["responsive"] - b[k] / b["responsive"]), chi2=c, p=p))
CT = pd.DataFrame(rows)
print(CT.round(4).to_string(index=False))
for rd in TONE:
    for fr in ("F1", "F2"):
        for pr in ("P1", "P2"):
            d = CT[(CT.tone == rd) & (((CT.period == "BL") & (CT.reading == fr)) | ((CT.period == "D1") & (CT.reading == f"{pr}-{fr}")))]
            print(f"{rd} {pr}-{fr}: comparisons with p < 0.05: {int((d.p < 0.05).sum())} of {len(d)}; reproduced {bool((d.p >= 0.05).all())}")

**Result: not reproduced.** Under every period, frame and tone reading at least one category differs
between groups at p < 0.05 by the paper's chi-square test: 2 of 8 comparisons under clock P1-F1 (BL
platform-only, shocked 24.7% against 33.3%, p = 0.0032; D1 mixed 24.2% against 29.8%, p = 0.043), 1
under clock P2-F1, 5 under clock P1-F2 and 6 under clock P2-F2 (D1 platform-only 20.2% against 33.8%,
p < 0.0001), and 3 to 6 of 8 with the paper's tone bouts.

In [ ]:
def pooled_area(day, feat, sign, event, half, select=None, roc=None, fe=None):
    roc = ROC if roc is None else roc
    fe = FE if fe is None else fe
    vals = {g: [] for g in GROUPS}
    per_mouse = []
    for s in S:
        if s[0] != day:
            continue
        feats, A, C = roc[(s, "All")]
        sel = (C[:, feats.index(feat)] == sign) if select is None else select(feats, C)
        if sel.sum() == 0:
            per_mouse.append(dict(group=s[1], mouse=s[2], cells=0, events=len(fe[s][1][event]), mean_area=np.nan))
            continue
        area, nev = pma.event_area(ca[s]["dff"][sel], CA_T[s], fe[s][1][event], half)
        vals[s[1]] += list(area[~np.isnan(area)])
        per_mouse.append(dict(group=s[1], mouse=s[2], cells=int(sel.sum()), events=nev, mean_area=np.nanmean(area)))
    a, b = np.array(vals["Shock"]), np.array(vals["NonShock"])
    t = stats.ttest_ind(a, b, equal_var=False)
    return dict(n_shock=len(a), n_ns=len(b), mean_shock=a.mean(), mean_ns=b.mean(), diff=a.mean() - b.mean(), t=t.statistic, p=t.pvalue), pd.DataFrame(per_mouse)

def show_area(label, *args, **kw):
    r, pm = pooled_area(*args, **kw)
    print(f"{label}: cells Shk {r['n_shock']} NS {r['n_ns']}; mean area Shk {r['mean_shock']:.3f} NS {r['mean_ns']:.3f} "
          f"(difference {r['diff']:.3f}); Welch t = {r['t']:.2f}, p = {r['p']:.3g}")
    return r, pm

def show_area_both(label, *args, **kw):
    """show_area under each tone reading; returns {reading: (result, per-mouse table)}."""
    return {rd: show_area(f"[{rd}] {label}", *args, roc=ROCS[rd][0], fe=ROCS[rd][1], **kw) for rd in TONE}

In [ ]:
rows = []
for rd in TONE:
    for pr, fr in READINGS:
        for g in GROUPS:
            f = {per: SUMS[rd][(pr, fr, per, g)] for per in PERDAY}
            rows.append(dict(tone=rd, reading=f"{pr}-{fr}", group=g, **{f"G1_{per}": f[per]["tone_any"] / f[per]["cells"] for per in PERDAY},
                             **{f"G2_{per}": f[per]["tone_only"] / f[per]["cells"] for per in PERDAY}))
TB = pd.DataFrame(rows)
print(TB.round(4).to_string(index=False))
for rd in TONE:
    for gr in ("G1", "G2"):
        for rg, d in TB[TB.tone == rd].groupby("reading", sort=False):
            ok = all((r[f"{gr}_BL"] > r[f"{gr}_D1"]) and (r[f"{gr}_BL"] > r[f"{gr}_D2"]) for _, r in d.iterrows())
            print(f"{rd} {gr} {rg}: BL greatest in both groups {ok}")
act = {rd: [] for rd in TONE}
for day in (1, 2):
    for sign, lab in ((1, "excited"), (-1, "suppressed")):
        for rd, (r, pm) in show_area_both(f"day {day} tone-onset {lab}", day, "toneonVector", sign, "tone_on", 5.0).items():
            act[rd].append(r["p"])
for rd in TONE:
    print(f"{rd}: activity similar (no p < 0.05): {all(p >= 0.05 for p in act[rd])}")

**Result: blocked (methods underspecified).** The activity part holds: tone-onset excited and
suppressed cells do not differ between groups on either day under either tone reading (p = 0.18 to
1.0) or any of the four shuffle seeds (smallest p = 0.083). The proportion part depends on the
readings: under G1 (any tone call) BL is not the greatest in both groups under any period or tone
reading (for example non-shocked clock P1-F1, BL 16.2% against D2 18.0%); under G2 (tone-only cells)
it is under P1-F1 with either tone reading (clock: shocked 10.7%, 8.8%, 8.9%; non-shocked 9.5%, 7.9%,
9.4%) and not under the other three. The claims file makes a claim whose readings disagree
methods_underspecified. The day 1 tone-onset cells number 33 and 37 shocked (43 and 38 with the paper's
bouts) against the paper's 61 and 52; no event window, cleaning of the platform bouts or shuffle seed
in the event-call check brings the counts to the paper's.

In [ ]:
rows = []
for rd in TONE:
    for pr, fr in READINGS:
        a, b = SUMS[rd][(pr, fr, "D2", "Shock")], SUMS[rd][(pr, fr, "D2", "NonShock")]
        for den, key in (("R1 responsive", "responsive"), ("R2 all cells", "cells")):
            c, p = pma.chi2_yates(a["mixed"], a[key], b["mixed"], b[key])
            ps, pn = 100 * a["mixed"] / a[key], 100 * b["mixed"] / b[key]
            rows.append(dict(tone=rd, reading=f"{pr}-{fr}", denominator=den, mixed_shock=a["mixed"], n_shock=a[key], pct_shock=ps,
                             mixed_ns=b["mixed"], n_ns=b[key], pct_ns=pn, chi2=c, p=p,
                             within_tol=bool(abs(ps - 51) <= 5.1 and abs(pn - 25) <= 2.5)))
MX = pd.DataFrame(rows)
print(MX.round(3).to_string(index=False))
print("Expected: 51% (tolerance 5.1) against 25% (2.5), chi-square 43.13, p < 0.0001")
print("Reproduced under any reading:", bool((MX.within_tol & (MX.p < 0.05) & (MX.pct_shock > MX.pct_ns)).any()))

**Result: not reproduced.** On day 2 the shocked mice have fewer mixed cells than the non-shocked mice
under every reading but one: among responsive cells 24.5% against 32.1% (clock P1-F1, p = 0.007), 25.0%
to 31.3% against 31.2% to 33.7% under the other clock readings and 25.9% to 32.2% against 28.2% to 33.1%
with the paper's tone bouts, where P2-F2 gives 28.6% against 28.2% (p = 0.94); among all cells, 12.0% to
16.7% against 14.5% to 18.3%. The paper reports 51% against 25%.

In [ ]:
A_P = show_area_both("day 2 platform-exit excited, exits", 2, "platextVector", 1, "plat_ext", 4.0)
R_P, PM_P = A_P["clock"]
print(PM_P.to_string(index=False))
print("Expected cells: 104 Shk (tolerance 10.4), 100 NS (10)")

# The same test with the platform-exit calls of three other shuffle seeds (platform exits do not
# depend on the tone reading), and with the cleaned platform bouts of the event-call check.
seed_rows = []
for sd in range(1, 4):
    roc_s = {}
    for s in S:
        if s[0] != 2:
            continue
        feats_s, A_s, C_s = ROC[(s, "All")]
        C_s = C_s.copy()
        rng = np.random.default_rng(zlib.crc32(f"exitseed{sd}{s[3]}".encode()))
        _, C_s[:, feats_s.index("platextVector")] = pma.roc_calls(ca[s]["dff"], FE[s][0]["platextVector"], rng)
        roc_s[(s, "All")] = (feats_s, A_s, C_s)
    r, _ = pooled_area(2, "platextVector", 1, "plat_ext", 4.0, roc=roc_s)
    seed_rows.append(dict(seed=sd, n_shock=r["n_shock"], n_ns=r["n_ns"], mean_shock=r["mean_shock"], mean_ns=r["mean_ns"], p=r["p"]))
print("\nOther shuffle seeds, clock reading:")
print(pd.DataFrame(seed_rows).round(4).to_string(index=False))

**Result: partially reproduced.** Day 2 platform-exit excited cells of shocked mice have a larger
post-exit area than those of non-shocked mice (12.09 against 6.43, Welch p = 0.044), but the cells number
77 and 84 against the paper's 104 and 100, outside the tolerance, under every event window and seed of
the event-call check (80 to 93 shocked cells). The effect is at the threshold. Eight runs are printed: the
main run; the shuffles of the paper-bout run (platform exits do not depend on the tone windows), p =
0.117, difference 4.30; three seeds of the exit calls alone (cell above), p = 0.068, 0.039 and 0.044,
differences 4.87, 5.82 and 5.31; and three seeds of the whole-session ROC (shuffle-seed cell below), p
= 0.043, 0.050 and 0.025, differences 5.26 to 6.45. Shocked cells are higher in all eight (difference
4.30 to 6.45), and p ≥ 0.05 in three (0.050, 0.068, 0.117).

In [ ]:
A_Q = show_area_both("day 2 platform-exit suppressed, exits", 2, "platextVector", -1, "plat_ext", 4.0)
R_Q, PM_Q = A_Q["clock"]
print(PM_Q.to_string(index=False))
print("Paper: 74 Shk and 82 NS cells")

**Result: reproduced.** Day 2 platform-exit suppressed cells (40 shocked, 65 non-shocked, against the
paper's 74 and 82) do not differ in post-exit area (-5.83 against -3.64, difference -2.19, Welch p =
0.56), nor with the paper's tone bouts (p = 0.46) or the three other seeds (p = 0.54 to 0.68).

In [ ]:
fz = {rd: [] for rd in TONE}
for sign, lab in ((1, "excited"), (-1, "suppressed")):
    for ev in ("frz_on", "frz_off"):
        for rd, (r, pm) in show_area_both(f"day 2 freezing-{lab}, {ev}", 2, "freezingVector", sign, ev, 3.0).items():
            fz[rd].append(r["p"])
for rd in TONE:
    print(f"{rd}: tests with p < 0.05: {sum(p < 0.05 for p in fz[rd])} of 4; at least one {any(p < 0.05 for p in fz[rd])}")

**Result: reproduced.** Three of the four day 2 freezing comparisons differ at p < 0.05 on the `clock`
run (172 and 150 excited, 280 and 198 suppressed cells, close to the paper's 171, 150, 283 and 198):
excited cells after freezing onset (7.36 against 10.58, p = 0.035) and offset (-7.38 against -3.91, p
= 0.015) are lower in shocked mice, and suppressed cells after offset are higher (3.57 against 0.39, p
= 0.0023). With the paper's tone bouts two of four do (p = 0.021 and 0.0061), and with the three other
seeds two or three.

In [ ]:
def mixed_sel(rule):
    def f(feats, C):
        K = pma.categories(C[:, :7])
        mixed = (K != 0).sum(1) >= 2
        if rule == "M1":
            return mixed & (C[:, feats.index("platextVector")] == 1)
        return mixed & (C[:, :7] == 1).any(1)
    return f
for rule in ("M1", "M2"):
    res = {rd: {} for rd in TONE}
    for day in (1, 2):
        for rd, (r, pm) in show_area_both(f"{rule} day {day} excited mixed cells, exits", day, "platextVector", 1, "plat_ext", 4.0, select=mixed_sel(rule)).items():
            res[rd][day] = r
    for rd in TONE:
        ok = res[rd][1]["diff"] > 0 and res[rd][1]["p"] < 0.05 and res[rd][2]["p"] >= 0.05
        print(f"{rule} {rd}: reproduced {ok}")

**Result: partially reproduced, resting on reading M2.** Mixed cells with any excited call (M2, the
code's split) are more active after platform exits in shocked mice on day 1 in every run (difference
2.6 to 3.6), but the difference reaches p < 0.05 only in the main run (250 and 181 cells, -1.21 against
-4.77, p = 0.049): p = 0.106 with the paper's tone bouts and 0.10 to 0.15 with the three other seeds.
Day 2 does not differ (p = 0.44 to 0.98). Mixed cells with an excited platform-exit call (M1) differ
on neither day in any run (day 1 p = 0.18 to 0.73).

In [ ]:
A_H = show_area_both("day 2 platform-exit excited, exits with no shock in the next 2 s", 2, "platextVector", 1, "plat_ext_noshock", 4.0)
for rd in TONE:
    print(f"{rd}: exits per session followed by a shock within 2 s:", {s[2]: len(ROCS[rd][1][s][1]["plat_ext_shock"]) for s in S if s[0] == 2})

**Result: reproduced.** With exits followed by a shock within 2 s left out (at most 2 per session) the
shocked cells remain higher (11.83 against 6.29, Welch p = 0.046), and with the three other seeds (p =
0.043, 0.045, 0.024). With the shuffles of the paper-bout run p = 0.143, so it shares the fragility of
fig1p-platform-exit-excited-day2.

## Escape (Figure S2)

The jumping and rearing labels are not deposited (data check), so the claim cannot be decided in
full. The escape part can be computed: the STAR Methods (Definitions of examined behaviors) define
escape as "the period of time when animals were off the platform during the final 2s of the tone".
Escape frames are the calcium frames off the platform in the last 2 s of each tone, escape onsets the
first frame of each such bout; escape cells are called by the whole-session ROC on that vector, and
the excited and suppressed cells are compared as in fig1p-platform-exit-excited-day2 around escape
onset, with the claims file's 3, 4 and 5 s windows, on both tone readings and four shuffle seeds.

In [ ]:
def escape_vector(b, cat, rd):
    tw = pma.tone_windows(b, rd)
    tw = tw[~np.isnan(tw[:, 0])]
    v = pma.window_vector(cat, tw[:, 1], -2.0, 0.0) & ~pma.to_ca(b, cat, "platform")
    return v, cat[pma.bouts(v)[:, 0]]

ESC = {}
for rd in TONE:
    for s in S:
        v, on = escape_vector(beh[s], CA_T[s], rd)
        for sd in range(4):
            rng = np.random.default_rng(zlib.crc32(f"escape{rd}{sd}{s[3]}".encode()))
            ESC[(rd, sd, s)] = (pma.roc_calls(ca[s]["dff"], v, rng)[1], on)
rows = []
for rd in TONE:
    for sd in range(4):
        for day in (1, 2):
            for sign in (1, -1):
                for half in (3.0, 4.0, 5.0):
                    vals, nev = {g: [] for g in GROUPS}, {g: 0 for g in GROUPS}
                    for s in S:
                        c, on = ESC[(rd, sd, s)]
                        if s[0] != day or (c == sign).sum() == 0:
                            continue
                        a, n = pma.event_area(ca[s]["dff"][c == sign], CA_T[s], on, half)
                        vals[s[1]] += list(a[~np.isnan(a)])
                        nev[s[1]] += n
                    a, b = np.array(vals["Shock"]), np.array(vals["NonShock"])
                    t = stats.ttest_ind(a, b, equal_var=False)
                    rows.append(dict(tone=rd, seed=sd, day=day, cells="excited" if sign == 1 else "suppressed", window_s=half,
                                     cells_Shk=len(a), cells_NS=len(b), escapes_Shk=nev["Shock"], escapes_NS=nev["NonShock"],
                                     area_Shk=a.mean(), area_NS=b.mean(), diff=a.mean() - b.mean(), welch_t=t.statistic, p=t.pvalue))
ESCT = pd.DataFrame(rows)
print("Escape cells, whole session; area after escape onset (z-scored DeltaF/F, s), Welch's t-test")
print(ESCT.round(4).to_string(index=False))
ex = ESCT[ESCT.cells == "excited"]
both = ex.groupby(["tone", "seed", "window_s"]).apply(lambda d: bool(((d["diff"] > 0) & (d.p < 0.05)).all()), include_groups=False)
print("\nEscape-excited part (shocked higher with p < 0.05 on both days) holds in", int(both.sum()), "of", len(both), "runs (tone reading, seed, window)")
print("Day 1 escape-excited p range:", f"{ex[ex.day == 1].p.min():.3f} to {ex[ex.day == 1].p.max():.3f}",
      "; day 2:", f"{ex[ex.day == 2].p.min():.3f} to {ex[ex.day == 2].p.max():.3f}")
sup = ESCT[ESCT.cells == "suppressed"]
print("Escape-suppressed cells, runs with p < 0.05:", int((sup.p < 0.05).sum()), "of", len(sup), f"(smallest p {sup.p.min():.3f})")


**Result: blocked (not runnable); a change to not reproduced is proposed.** Jumping and rearing cannot
be scored, so the claim stays blocked as verified. The escape part, which the criterion needs to hold,
fails in every run: escape-excited cells of shocked mice are not more active after escape onset than
those of non-shocked mice on day 1 under any window, tone reading or seed (p = 0.40 to 0.99 on the
`clock` windows, 0.45 to 0.99 on the paper's bouts), and on day 2 the difference is positive but p ≥
0.067 in every run. Escape-suppressed cells do not differ in any run (smallest p 0.17). As the criterion is met only when all parts hold, the claim fails whatever the jumping
part gives, so the record proposes not reproduced for a third verification round.

## Platform entries and exits during the tone and the ITI (Figure 2)

In [ ]:
def folded(day, feat, roc):
    rows = []
    for s in S:
        if s[0] != day:
            continue
        feats, A, C = roc[(s, "All")]
        j = feats.index(feat)
        sel = C[:, j] != 0
        v = np.where(C[sel, j] == 1, A[sel, j], 1 - A[sel, j])
        rows += [dict(group=s[1], mouse=s[2], value=x) for x in v]
    return pd.DataFrame(rows)

def lsd_anova(d):
    """Ordinary two-way ANOVA (group by epoch) and Fisher's LSD from its residual mean square."""
    m = smf.ols("value ~ C(group) * C(epoch)", data=d).fit()
    mse, df = m.mse_resid, m.df_resid
    g = d.groupby(["group", "epoch"]).value
    mean, n = g.mean(), g.count()
    def lsd(a, b):
        t = (mean[a] - mean[b]) / np.sqrt(mse * (1 / n[a] + 1 / n[b]))
        return t, 2 * stats.t.sf(abs(t), df)
    return mean, n, lsd

FIG2 = {}
for rd in TONE:
    for day in (1, 2):
        for ev in ("Platent", "Platext"):
            roc = ROCS[rd][0]
            cells = pd.concat([folded(day, f"tone{ev}Vector", roc).assign(epoch="tone"), folded(day, f"iti{ev}Vector", roc).assign(epoch="iti")])
            mice = cells.groupby(["group", "mouse", "epoch"]).value.agg(["mean", "count"]).reset_index()
            mice = mice[mice["count"] >= 3].rename(columns={"mean": "value"})
            for H, d in (("H1", cells), ("H2", mice)):
                mean, n, lsd = lsd_anova(d)
                FIG2[(rd, day, ev, H)] = dict(mean=mean, n=n, lsd=lsd)
                print(f"[{rd}] day {day} {ev} {H}: means " + ", ".join(f"{k[0]} {k[1]} {mean[k]:.3f} (n {n[k]})" for k in mean.index))
                for gname in GROUPS:
                    t, p = lsd((gname, "tone"), (gname, "iti"))
                    print(f"    {gname} tone vs ITI: t = {t:.2f}, p = {p:.3g}")
                t, p = lsd(("Shock", "tone"), ("NonShock", "tone"))
                print(f"    tone, Shock vs NonShock: t = {t:.2f}, p = {p:.3g}")

In [ ]:
# Shuffle seeds. The calls of cells near the 2.5% and 97.5% bounds change with the seed of the
# 1000 circular shuffles. The whole-session ROC (clock reading) is run with three more seeds and
# the mean-activity and Figure 2 tests are repeated on each.
SEED_ROC = {0: ROC}
for sd in (1, 2, 3):
    SEED_ROC[sd], _ = run_roc("clock", which={"All"}, tag=f"seed{sd}")
rows = []
for sd, roc in SEED_ROC.items():
    r = dict(seed=sd)
    for key, args, kw in (("fig1p", (2, "platextVector", 1, "plat_ext", 4.0), {}),
                          ("fig1q", (2, "platextVector", -1, "plat_ext", 4.0), {}),
                          ("figs1h", (2, "platextVector", 1, "plat_ext_noshock", 4.0), {}),
                          ("fig1r_M1_d1", (1, "platextVector", 1, "plat_ext", 4.0), {"select": mixed_sel("M1")}),
                          ("fig1r_M1_d2", (2, "platextVector", 1, "plat_ext", 4.0), {"select": mixed_sel("M1")}),
                          ("fig1r_M2_d1", (1, "platextVector", 1, "plat_ext", 4.0), {"select": mixed_sel("M2")}),
                          ("fig1r_M2_d2", (2, "platextVector", 1, "plat_ext", 4.0), {"select": mixed_sel("M2")})):
        a, _ = pooled_area(*args, roc=roc, **kw)
        r[f"{key}_cells"] = f"{a['n_shock']}/{a['n_ns']}"
        r[f"{key}_diff"] = round(a["diff"], 2)
        r[f"{key}_p"] = round(a["p"], 4)
    fzp = []
    for sign in (1, -1):
        for ev in ("frz_on", "frz_off"):
            fzp.append(pooled_area(2, "freezingVector", sign, ev, 3.0, roc=roc)[0]["p"])
    r["fig1pq_tests_p<0.05"] = sum(p < 0.05 for p in fzp)
    tp = [pooled_area(day, "toneonVector", sign, "tone_on", 5.0, roc=roc)[0]["p"] for day in (1, 2) for sign in (1, -1)]
    r["disc_tone_activity_min_p"] = round(min(tp), 3)
    rows.append(r)
SEEDS = pd.DataFrame(rows).set_index("seed").T
print(SEEDS.to_string())

rows = []
for sd, roc in SEED_ROC.items():
    for day in (1, 2):
        for ev in ("Platent", "Platext"):
            cells = pd.concat([folded(day, f"tone{ev}Vector", roc).assign(epoch="tone"), folded(day, f"iti{ev}Vector", roc).assign(epoch="iti")])
            mice = cells.groupby(["group", "mouse", "epoch"]).value.agg(["mean", "count"]).reset_index()
            mice = mice[mice["count"] >= 3].rename(columns={"mean": "value"})
            for H, d in (("H1", cells), ("H2", mice)):
                mean, n, lsd = lsd_anova(d)
                out = dict(seed=sd, day=day, event=ev, reading=H)
                for gname in GROUPS:
                    out[f"{gname} tone-ITI p"] = lsd((gname, "tone"), (gname, "iti"))[1]
                out["Shk tone"], out["NS tone"] = mean[("Shock", "tone")], mean[("NonShock", "tone")]
                out["Shk-NS tone p"] = lsd(("Shock", "tone"), ("NonShock", "tone"))[1]
                out["mice Shk/NS"] = f"{n[('Shock', 'tone')]}/{n[('NonShock', 'tone')]}" if H == "H2" else ""
                rows.append(out)
print("\nFigure 2 tests by shuffle seed (clock reading):")
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
for rd in TONE:
    for H in ("H1", "H2"):
        ok = True
        for ev in ("Platent", "Platext"):
            f = FIG2[(rd, 1, ev, H)]
            for gname in GROUPS:
                t, p = f["lsd"]((gname, "tone"), (gname, "iti"))
                hit = f["mean"][(gname, "tone")] > f["mean"][(gname, "iti")] and p < 0.05
                ok &= hit
                print(f"[{rd}] {H} day 1 {ev} {gname}: tone {f['mean'][(gname, 'tone')]:.3f} ITI {f['mean'][(gname, 'iti')]:.3f}, LSD p = {p:.3g}, holds {hit}")
        print(f"[{rd}] {H}: reproduced {ok}")

**Result: reproduced.** On day 1 the folded auROC of entry and exit cells is higher for events during
the tone than during the ITI in both groups, pooled over cells (H1, for example shocked entries 0.706
against 0.640, Fisher's LSD p < 1e-30) and over per-mouse scores (H2, p = 2.2e-06 to 0.0017), under
both tone readings and all four shuffle seeds (largest p = 0.0049).

In [ ]:
for rd in TONE:
    for H in ("H1", "H2"):
        ok = True
        for ev in ("Platent", "Platext"):
            f2, f1 = FIG2[(rd, 2, ev, H)], FIG2[(rd, 1, ev, H)]
            t, p = f2["lsd"](("Shock", "tone"), ("NonShock", "tone"))
            hit = f2["mean"][("Shock", "tone")] > f2["mean"][("NonShock", "tone")] and p < 0.05
            up = f2["mean"][("Shock", "tone")] > f1["mean"][("Shock", "tone")]
            ok &= hit and up
            print(f"[{rd}] {H} {ev}: day 2 Shock tone {f2['mean'][('Shock', 'tone')]:.3f} NS tone {f2['mean'][('NonShock', 'tone')]:.3f} (LSD p = {p:.3g}); "
                  f"Shock day 1 tone {f1['mean'][('Shock', 'tone')]:.3f}; holds {hit and up}")
        print(f"[{rd}] {H}: reproduced {ok}")

**Result: partially reproduced, resting on reading H1.** Exits hold in every run: day 2 shocked tone
values exceed non-shocked ones (H1 0.783 against 0.718, LSD p = 5.4e-13; p < 1e-4 under all seeds and
2.6e-07 with the paper's bouts) and the shocked day 1 values (0.736). Entries hold only in the main
run (H1 0.744 against 0.728, p = 0.034); with the three other seeds p = 0.11 to 0.58, and with the
paper's bouts p = 0.14. Over per-mouse scores (H2) entries fail in every run (p = 0.086 to 0.21).

In [ ]:
for rd in TONE:
    f = FIG2[(rd, 2, "Platent", "H2")]
    t, p = f["lsd"](("Shock", "tone"), ("NonShock", "tone"))
    print(f"[{rd}] day 2 entries, modulation score (mean folded auROC), mice with at least 3 cells: n {f['n'].to_dict()}")
    print(f"[{rd}] Shock tone {f['mean'][('Shock', 'tone')]:.3f}, NonShock tone {f['mean'][('NonShock', 'tone')]:.3f}; LSD t = {t:.2f}, p = {p:.3g}")
print("Paper legend: day 2 Shk N=7, NS N=7")

**Result: not reproduced.** The day 2 entry modulation score of shocked mice during the tone is 0.755
against 0.710 for non-shocked mice (LSD p = 0.086), with 6 and 5 mice having at least 3 cells, where the
paper had 7 and 7. It is not significant with the three other seeds (p = 0.13 to 0.21) or the paper's
tone bouts (p = 0.175).

In [ ]:
for rd in TONE:
    f = FIG2[(rd, 2, "Platext", "H2")]
    t, p = f["lsd"](("Shock", "tone"), ("NonShock", "tone"))
    print(f"[{rd}] day 2 exits, modulation score, mice with at least 3 cells: n {f['n'].to_dict()}")
    print(f"[{rd}] Shock tone {f['mean'][('Shock', 'tone')]:.3f}, NonShock tone {f['mean'][('NonShock', 'tone')]:.3f}; LSD t = {t:.2f}, p = {p:.3g}")
print("Paper legend: day 2 Shk N=6, NS N=7")

**Result: reproduced.** The day 2 exit modulation score during the tone is 0.807 in shocked against
0.705 in non-shocked mice (LSD p = 0.0059), with 4 and 4 mice having at least 3 cells, where the paper
had 6 and 7; p = 0.0031 to 0.0059 with the three other seeds and 0.018 with the paper's tone bouts.

## Decoding trial outcome on day 1 (Figure 3)

Per mouse, day 1 trials 4 to 12, one LIBSVM RBF decoder per 200 ms bin from 20 s before to
50 s after tone onset (a_SVM_permousev3ASF.m, as the claims file gives it). Cell sets for
Figure 3B and 3C come from the ROC calls of the BL period (F1) and the D1 period (P1-F1). Each
result is shown for three cohorts (all 9 shocked mice; without F1 and F2, the 7 shocked mice imaged
on both days; without the mice whose leave-one-out training folds hold one class) and, for all
cells, on the paper's tone bouts too.

In [ ]:
EPOCHS = {"pre-cue": (-5, 0), "cue-onset": (0, 5), "pre-shock": (23, 28), "shock": (28, 33), "post-shock": (34, 38), "pre-shock-code": (22, 27)}
METHODS_EPOCHS = ["cue-onset", "pre-shock", "shock", "post-shock"]

def cellsets(s, roc, pr="P1", fr="F1"):
    kb = pma.categories(roc[(s, f"BL-{fr}")][2][:, :7])
    kd = pma.categories(roc[(s, f"D1-{pr}-{fr}")][2][:, :7])
    bl, d1 = (kb != 0).any(1), (kd != 0).any(1)
    return {"K1": bl | d1, "K2": d1, "nonmod": ~(bl | d1)}

# Cell sets. The base sets (K1, K2, nonmod) use BL F1 and D1 P1-F1; the sets of the other three
# period and frame readings are named with the reading (for example "K1 P2-F2"). On the clock
# windows every reading is decoded; on the paper's bouts all cells and the base sets.
SET_READINGS = {"clock": READINGS, "paper": [("P1", "F1")]}
def set_name(k, pr, fr):
    return k if (pr, fr) == ("P1", "F1") else f"{k} {pr}-{fr}"
CELLSETS = {rd: ["all"] + [set_name(k, pr, fr) for pr, fr in SET_READINGS[rd] for k in ("K1", "K2", "nonmod")] for rd in TONE}

# Shuffled-label control: for each mouse on the clock windows, all cells are decoded again with the
# trial outcomes permuted (the mouse's number of successes kept), N_PERM times.
N_PERM = 5

jobs, keys, DEC_N, same, seen = [], [], [], {}, {}
for rd in TONE:
    T = TT[rd]
    for s in S:
        if s[0] != 1:
            continue
        lab = T[(T.day == 1) & (T.group == s[1]) & (T.mouse == s[2])].sort_values("trial").success.values[3:12]
        on = pma.tone_windows(beh[s], rd)[3:12, 0]
        X, BT = pma.decode_inputs(ca[s]["dff"], CA_T[s], on)
        sets = {"all": np.ones(X.shape[2], bool)}
        for pr, fr in SET_READINGS[rd]:
            sets.update({set_name(k, pr, fr): v for k, v in cellsets(s, ROCS[rd][0], pr, fr).items()})
        DEC_N.append(dict(tone=rd, group=s[1], mouse=s[2], successes=int(lab.sum()), failures=int((lab == 0).sum()),
                          **{k: int(sets[k].sum()) for k in ("all", "K1", "K2", "nonmod")}))
        todo = [(k, sel, lab, k) for k, sel in sets.items()]
        if rd == "clock":
            for i in range(N_PERM):
                perm = np.random.default_rng(zlib.crc32(f"null{s[3]}{i}".encode())).permutation(lab)
                todo.append((f"null{i}", sets["all"], perm, "all"))
        for k, sel, y, seedk in todo:
            if sel.sum() == 0:
                continue
            job = (X[:, :, sel], y, zlib.crc32(f"{s[3]}{seedk}".encode()) % 100000)
            # A decoder whose inputs and labels equal one already queued for the mouse is not run again.
            h = (s, zlib.crc32(job[0].tobytes()), tuple(y))
            if h in seen:
                same[(rd, s, k)] = seen[h]
                continue
            seen[h] = (rd, s, k)
            jobs.append(job)
            keys.append((rd, s, k))
t0 = time.time()
# One OpenMP thread per LIBSVM process and four processes; the predictions are the same as one
# process's (each bin's folds are seeded).
os.environ["OMP_NUM_THREADS"] = "1"
order = sorted(range(len(jobs)), key=lambda i: -jobs[i][0].size)
with ProcessPoolExecutor(4, mp_context=mp.get_context("spawn")) as ex:
    PRED = {keys[i]: P for i, P in zip(order, ex.map(pma.decode_job, [jobs[i] for i in order]))}
PRED.update({k: PRED[v] for k, v in same.items()})
LAB = {k: j[1] for k, j in zip(keys, jobs)}
LAB.update({k: LAB[v] for k, v in same.items()})
print(f"paper reading: decoded again for all cells (inputs or labels differ from the clock reading's):",
      sorted(s[2] for (rd, s, k) in keys if rd == "paper" and k == "all"))
DN = pd.DataFrame(DEC_N)
print(f"{len(jobs)} decoders run (tone reading, mouse and cell set or label permutation), {len(same)} taken from an identical one, "
      f"{len(BT)} bins each, in {time.time() - t0:.0f} s")
print(DN.to_string(index=False))

# Cohorts. The Figure 3 legend gives 7 shocked mice; 9 are imaged on day 1 and the paper does not
# say which two were left out. "no F1 F2" drops the two mice with no day 2 recording; "two-class"
# drops every mouse with fewer than two trials of either outcome, whose leave-one-out training
# folds hold a single class.
def cohort_drop(rd, cohort):
    d = DN[DN.tone == rd]
    if cohort == "all":
        return set()
    if cohort == "no F1 F2":
        return {"F1", "F2"}
    return set(d[(d.successes < 2) | (d.failures < 2)].mouse)
COHORTS = ("all", "no F1 F2", "two-class")
for rd in TONE:
    for co in COHORTS:
        print(f"{rd}, cohort {co}: left out {sorted(cohort_drop(rd, co))}")

def epoch_table(rd, cellset, positive, norm, drop=()):
    rows = []
    for (r, s, k), P in PRED.items():
        if r != rd or k != cellset or s[2] in drop:
            continue
        f1 = pma.normalise(pma.f1_scores(P, LAB[(r, s, k)], positive), norm)
        rows.append(dict(group=s[1], mouse=s[2], **{e: f1[(BT >= a - 1e-9) & (BT < b - 1e-9)].mean() for e, (a, b) in EPOCHS.items()}, trace=f1))
    return pd.DataFrame(rows)

def group_test(E, epochs=METHODS_EPOCHS):
    long = E.melt(id_vars=["group", "mouse"], value_vars=epochs, var_name="epoch", value_name="f1")
    aov = pg.mixed_anova(long, dv="f1", within="epoch", between="group", subject="mouse", correction=True)
    sid = {}
    for e in epochs:
        a, b = E[E.group == "Shock"][e], E[E.group == "NonShock"][e]
        sid[e] = (a.mean(), b.mean(), pma.sidak(stats.ttest_ind(a, b).pvalue, len(epochs)))
    return aov, sid

DEC = {}
for rd in TONE:
    for co in COHORTS:
        drop = cohort_drop(rd, co)
        for cs in CELLSETS[rd]:
            for pos in ("success", "failure"):
                for norm in ("N1", "N2"):
                    E = epoch_table(rd, cs, pos, norm, drop)
                    aov, sid = group_test(E)
                    _, sid_code = group_test(E, ["cue-onset", "pre-shock-code", "shock", "post-shock"])
                    DEC[(rd, co, cs, pos, norm)] = dict(E=E, aov=aov, sid=sid, sid_code=sid_code)
rows = []
for (rd, co, cs, pos, norm), d in DEC.items():
    if cs not in ("all", "K1", "K2", "nonmod"):
        continue                                   # the other readings' sets are shown under Figure 3B and 3C
    a = d["aov"].set_index("Source")
    rows.append(dict(tone=rd, cohort=co, cells=cs, reading=f"C{1 if pos == 'success' else 2} {norm}", Shk=int((d["E"].group == "Shock").sum()),
                     NS=int((d["E"].group == "NonShock").sum()), group_F=a.loc["group", "F"], group_p=a.loc["group", "p_unc"],
                     **{f"{e} Shk/NS (p)": f"{v[0]:.3f}/{v[1]:.3f} ({v[2]:.3g})" for e, v in d["sid"].items()},
                     code_preshock_p=d["sid_code"]["pre-shock-code"][2]))
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
def c_lab(pos, norm):
    return f"C{1 if pos == 'success' else 2} {norm}"
for rd in TONE:
    for co in COHORTS:
        for pos in ("success", "failure"):
            for norm in ("N1", "N2"):
                d = DEC[(rd, co, "all", pos, norm)]
                ok = all(d["sid"][e][0] > d["sid"][e][1] and d["sid"][e][2] < 0.05 for e in ("pre-shock", "shock"))
                okc = all(d["sid_code"][e][0] > d["sid_code"][e][1] and d["sid_code"][e][2] < 0.05 for e in ("pre-shock-code", "shock"))
                print(f"[{rd}, {co}] {c_lab(pos, norm)}: pre-shock {d['sid']['pre-shock'][0]:.3f} vs {d['sid']['pre-shock'][1]:.3f} (p {d['sid']['pre-shock'][2]:.3g}), "
                      f"shock {d['sid']['shock'][0]:.3f} vs {d['sid']['shock'][1]:.3f} (p {d['sid']['shock'][2]:.3g}); holds {ok}; with the code's 22-27 s epoch {okc}")
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, pos in zip(axs, ("failure", "success")):
    E = DEC[("clock", "all", "all", pos, "N1")]["E"]
    for g, col in zip(GROUPS, ("tab:red", "tab:gray")):
        tr = np.vstack(E[E.group == g].trace)
        ax.plot(BT, tr.mean(0), color=col, label=g)
    ax.set_title(f"all cells, positive class {pos}, N1, clock")
    ax.set_xlabel("s from tone onset")
axs[0].legend()
plt.show()
print("Plotted: across-mouse means of min-max normalized F1, range 0 to 1 by construction")

In [ ]:
# Shuffled-label control (clock windows, all cells). The class that is the majority in a mouse's
# nine trials gets the higher F1, and shocked mice have more successes, so a group difference could
# come from the share of successes. Each mouse's decoder is run again with its trial outcomes
# permuted (N_PERM times, the number of successes kept); the null value of an epoch is the mean
# over the permutations. Shown: the groups' null values, and the observed minus the null value.
def null_table(pos, norm, drop=()):
    rows = []
    for s in S:
        if s[0] != 1 or s[2] in drop:
            continue
        ep = []
        for i in range(N_PERM):
            k = ("clock", s, f"null{i}")
            f1 = pma.normalise(pma.f1_scores(PRED[k], LAB[k], pos), norm)
            ep.append({e: f1[(BT >= a - 1e-9) & (BT < b - 1e-9)].mean() for e, (a, b) in EPOCHS.items()})
        rows.append(dict(group=s[1], mouse=s[2], **pd.DataFrame(ep).mean().to_dict()))
    return pd.DataFrame(rows)

def sidak_rows(E, label, epochs=("pre-cue", "pre-shock", "shock")):
    out = {}
    for e in epochs:
        a, b = E[E.group == "Shock"][e], E[E.group == "NonShock"][e]
        m = 5 if e == "pre-cue" else len(METHODS_EPOCHS)
        out[e] = f"{a.mean():.3f}/{b.mean():.3f} (p {pma.sidak(stats.ttest_ind(a, b).pvalue, m):.3g})"
    return dict(values=label, **out)

NULL = {}
rows = []
for co in COHORTS:
    drop = cohort_drop("clock", co)
    for pos in ("success", "failure"):
        for norm in ("N1", "N2"):
            obs = DEC[("clock", co, "all", pos, norm)]["E"].set_index("mouse")
            nul = null_table(pos, norm, drop).set_index("mouse")
            dif = obs[list(EPOCHS)].sub(nul.loc[obs.index, list(EPOCHS)]).assign(group=obs.group).reset_index()
            NULL[(co, pos, norm)] = (nul, dif)
            for lab, E in (("observed", obs.reset_index()), ("shuffled labels", nul.reset_index()), ("observed - shuffled", dif)):
                rows.append(dict(cohort=co, reading=c_lab(pos, norm), **sidak_rows(E, lab)))
print("Shock/NonShock means (Sidak p; pre-cue over 5 epochs, as in the pre-cue claim's report), clock windows, all cells")
print(pd.DataFrame(rows).to_string(index=False))
for co in COHORTS:
    nul, dif = NULL[(co, "success", "N1")]
    holds = all(dif[dif.group == "Shock"][e].mean() > dif[dif.group == "NonShock"][e].mean() and
                pma.sidak(stats.ttest_ind(dif[dif.group == "Shock"][e], dif[dif.group == "NonShock"][e]).pvalue, 4) < 0.05 for e in ("pre-shock", "shock"))
    pc = dif[dif.group == "Shock"]["pre-cue"].mean() > dif[dif.group == "NonShock"]["pre-cue"].mean()
    print(f"C1 N1, {co}: above the shuffled-label values, Shock higher with Sidak p < 0.05 in pre-shock and shock {holds}; Shock higher before the tone {pc}")
obs = DEC[("clock", "all", "all", "success", "N1")]["E"].set_index("mouse")
nul = NULL[("all", "success", "N1")][0]
pm = DN[DN.tone == "clock"].set_index("mouse")[["group", "successes"]]
for e in ("pre-cue", "pre-shock", "shock"):
    pm[f"{e} obs"], pm[f"{e} null"] = obs[e], nul[e]
print("\nPer mouse, C1 N1, clock, all cells: successes of 9 and the epoch values, observed and with shuffled labels")
print(pm.round(3).to_string())
for e in ("pre-shock", "shock"):
    r = stats.pearsonr(pm.successes, pm[f"{e} null"])
    print(f"Correlation of successes with the shuffled-label {e} value over 16 mice: r = {r[0]:.2f}, p = {r[1]:.3g}")


**Result: reproduced, resting on the Methods' coding (success positive, min-max normalized F1) and the
`clock` tone windows.** With C1 N1 shocked mice decode better than non-shocked mice in the pre-shock
epoch (0.489 against 0.108, Sidak p = 0.036) and the shock epoch (0.658 against 0.099, p = 0.012), and
with the code's 22 to 27 s epoch too. This holds in all three cohorts (without F1 and F2, 7 and 7
mice: p = 9.9e-05 and 1.0e-06; two-class folds only: p = 0.0012 and 2.2e-05). Under the other three
codings the pre-shock difference is not significant in any cohort. With the paper's tone bouts C1 N1
holds without F1 and F2 (p = 0.012 and 0.0032) and fails with all 9 mice (p = 0.15 and 0.064) and in
the two-class cohort (p = 0.17 and 0.057).

The shuffled-label control does not support the pre-shock part. With permuted outcomes the shocked
mice score as high as with the real ones (C1 N1, pre-shock 0.476 against 0.489 observed; shock 0.482
against 0.658), while the non-shocked mice score above their observed values (0.230 against 0.108).
Above the shuffled-label values the pre-shock difference is not significant in any cohort or coding
(C1 N1: 0.013 against -0.122, Sidak p = 0.57 with all mice, 0.67 and 0.55 in the other cohorts). The
shock epoch is higher in shocked mice above the shuffled values too, at the threshold under C1 N1 (p =
0.054, 0.046 and 0.048) and clearly under C2 (C2 N1, p = 0.0018 to 0.013). So the shocked mice decode
the outcome in the shock epoch, and in the pre-shock epoch they decode it no better than with shuffled
labels; the C1 N1 group difference there comes from the non-shocked mice scoring below their own
shuffled values. The verified outcome is kept, and the record proposes partially reproduced.

In [ ]:
for rd in TONE:
    for co in COHORTS:
        for pos in ("success", "failure"):
            for norm in ("N1", "N2"):
                E = DEC[(rd, co, "all", pos, norm)]["E"]
                a, b = E[E.group == "Shock"]["pre-cue"], E[E.group == "NonShock"]["pre-cue"]
                p = pma.sidak(stats.ttest_ind(a, b).pvalue, 5)
                print(f"[{rd}, {co}] {c_lab(pos, norm)}: pre-cue (-5 to 0 s) Shk {a.mean():.3f} NS {b.mean():.3f}, Shk higher {a.mean() > b.mean()}; Sidak p (5 epochs) {p:.3g}")

**Result: reproduced, resting on the same readings.** Under C1 N1 on the `clock` windows shocked mice
decode better than non-shocked mice from 5 s before the tone to its onset in every cohort (0.481
against 0.134 with all mice, 0.619 against 0.134 without F1 and F2; the Sidak p, which the criterion
does not use, is 0.15 and 0.0065), and with the paper's bouts (0.377 against 0.129). Under C1 N2 and
C2 N2 shocked mice are not higher in any cohort. In the shuffled-label control the shocked mice's
pre-cue value equals their shuffled-label value (0.481 against 0.480), and the non-shocked mice's is
below theirs (0.134 against 0.262): above the shuffled values shocked mice are still higher under C1 N1
(0.001 against -0.128, p = 0.78) and C2 N1, and lower under C1 N2 and C2 N2. The criterion, a comparison
of means, is met; that shocked mice decode the outcome before the tone better than chance is not.

In [ ]:
for rd in TONE:
    for co in COHORTS:
        for pos in ("success", "failure"):
            for norm in ("N1", "N2"):
                E = DEC[(rd, co, "all", pos, norm)]["E"]
                sh = E[E.group == "Shock"]
                means = sh[METHODS_EPOCHS].mean()
                tr = np.vstack(sh.trace).mean(0)
                w = (BT >= 0) & (BT < 38)
                tpk = BT[w][np.argmax(tr[w])]
                print(f"[{rd}, {co}] {c_lab(pos, norm)}: Shock epoch means " + ", ".join(f"{k} {v:.3f}" for k, v in means.items())
                      + f"; P1 pre-shock highest {means.idxmax() == 'pre-shock'}; P2 peak bin at {tpk:.1f} s, in 23-28 s {23 <= tpk < 28}")

**Result: not reproduced.** In shocked mice the decoder peaks in the shock epoch under every coding,
cohort and tone reading: the highest bin of the mean trace is at 32.4 to 32.6 s after tone onset, and
the shock epoch has the highest epoch mean (clock C1 N1, shock 0.658 against pre-shock 0.489).

In [ ]:
def nsig(d):
    return sum(d["sid"][e][0] > d["sid"][e][1] and d["sid"][e][2] < 0.05 for e in METHODS_EPOCHS)

def compare_sets(kind, sets, lower):
    """E1 and E2 of fig3b (lower=False) or fig3c (lower=True) for each cell set against all cells."""
    rows = []
    for rd in TONE:
        for co in COHORTS:
            for cs in sets:
                if (rd, co, cs, "success", "N1") not in DEC:
                    continue
                for pos in ("success", "failure"):
                    for norm in ("N1", "N2"):
                        d, a = DEC[(rd, co, cs, pos, norm)], DEC[(rd, co, "all", pos, norm)]
                        em, am = d["E"].set_index("mouse"), a["E"].set_index("mouse")
                        sh = [m for m in em.index if em.loc[m, "group"] == "Shock" and m in am.index]
                        e1 = {e: (em.loc[sh, e].mean(), am.loc[sh, e].mean()) for e in ("pre-shock", "shock")}
                        E1 = all((v[0] < v[1]) if lower else (v[0] > v[1]) for v in e1.values())
                        E2 = (nsig(d) < nsig(a)) if lower else (nsig(d) >= nsig(a))
                        rows.append(dict(tone=rd, cohort=co, cells=cs, reading=c_lab(pos, norm),
                                         preshock=f"{e1['pre-shock'][0]:.3f} vs {e1['pre-shock'][1]:.3f}", shock=f"{e1['shock'][0]:.3f} vs {e1['shock'][1]:.3f}",
                                         E1=E1, sig_epochs=f"{nsig(d)} vs {nsig(a)}", E2=E2))
    R = pd.DataFrame(rows)
    print(f"{kind}: Shock mean normalized F1 with the cell set vs all cells, and epochs with Shock significantly higher (set vs all cells)")
    print(R.to_string(index=False))
    s = R[R.reading == "C1 N1"].copy()
    s["set"] = s.cells.str.split(" ").str[0]
    s["period_reading"] = s.cells.str.split(" ").str[1].fillna("P1-F1")
    print(f"\n{kind}, C1 N1: E1 and E2 by tone reading, cell set and period reading, over the three cohorts")
    print(s.groupby(["tone", "set", "period_reading"]).agg(E1_holds=("E1", "sum"), E2_holds=("E2", "sum"), cohorts=("E1", "size")).reset_index().to_string(index=False))
    return R

FIG3B = compare_sets("Figure 3B", [set_name(k, pr, fr) for pr, fr in READINGS for k in ("K1", "K2")], lower=False)


**Result: not reproduced, resting on the Methods' coding and the `clock` tone windows.** Under C1 N1,
decoding from modulated cells lowers the shocked mice's pre-shock and shock values (K1: 0.383 and 0.626
against 0.489 and 0.658 with all cells; K2: 0.442 and 0.582) and leaves fewer epochs where shocked mice
are significantly higher (0 against 3 with all cells), so E1 and E2 both fail, for K1 and K2, in all
three cohorts. The same holds for the cell sets of the other three period and frame readings on the
`clock` windows: E1 and E2 fail for K1 and K2 under P1-F2, P2-F1 and P2-F2 in every cohort (with P1-F1,
24 of 24 cases). On the paper's tone bouts (P1-F1 sets) E1 holds for K1 and K2 in all three cohorts (for
example K1, all mice, 0.487 and 0.660 against 0.407 and 0.561) and E2 in two, all mice and two-class,
where the all-cell decoder has no significant epoch; without F1 and F2, E2 fails (2 and 0 against 3). So the outcome
rests on the `clock` windows too. Under the six other combinations of coding and cell set on the `clock`
windows E2 holds in most and E1 in two (K2 with C2).

In [ ]:
FIG3C = compare_sets("Figure 3C", [set_name("nonmod", pr, fr) for pr, fr in READINGS], lower=True)


**Result: blocked (methods underspecified).** Under C1 N1 on the `clock` windows non-modulated cells
give fewer significant epochs than all cells (0 against 3, E2 holds) but a higher pre-shock value (0.499
against 0.489, E1 fails) in all three cohorts, and the same under the other three period and frame
readings (E1 fails and E2 holds in all 12 cases of the four
readings and three cohorts). On the paper's tone bouts both fail in every cohort.
The claims file makes a disagreement between E1 and E2 methods_underspecified.

## Rescorla-Wagner models (Figure 4B)

In [ ]:
def rw_inputs(mouse, trials, seed, rd="clock"):
    T = TT[rd]
    d = T[(T.group == "Shock") & (T.mouse == mouse)].sort_values(["day", "trial"])
    d1 = d[d.day == 1]
    if seed == "S1":
        v0 = d1[d1.trial <= 3].platform.mean()
    else:
        s = [x for x in S if x[0] == 1 and x[1] == "Shock" and x[2] == mouse][0]
        b = beh[s]
        first = pma.tone_windows(b, rd)[0, 0]
        t = b["cam_t"][:len(b["platform"])]
        v0 = b["platform"][:len(t)][t < first].mean()
    if trials == "T1":
        d = d[~((d.day == 1) & (d.trial <= 3))]
    d = d.dropna(subset=["platform", "success"])
    return d.platform.values, d.success.values, v0

# U1, the claims file's update (R(t+1), from the deposited script); U2, the STAR Methods'
# formula (R_t). "multi" starts the fit from five points and keeps the best; the code starts at [1, 1].
STARTS = {"code": ((1, 1),), "multi": ((1, 1), (0.5, 0.5), (0.1, 0.9), (0.9, 0.1), (0.2, 0.2))}
RW, rw_rows = {}, []
for rd in TONE:
    for form in ("U1", "U2"):
        for st in ("code", "multi"):
            if st == "multi" and rd == "paper":
                continue
            for trials in ("T1", "T2"):
                for seed in ("S1", "S2"):
                    rows = []
                    for mouse in sorted({s[2] for s in S if s[1] == "Shock" and s[0] == 1}):
                        data, succ, v0 = rw_inputs(mouse, trials, seed, rd)
                        row = dict(mouse=mouse, trials=len(data), v0=v0)
                        for model in ("shock", "avoid", "hybrid"):
                            a, q, pred = pma.rw_fit(data, succ, v0, model, form, STARTS[st])
                            nb_ = len(data) // 3
                            res = [(pred[3 * k:3 * k + 3].mean() - data[3 * k:3 * k + 3].mean()) ** 2 for k in range(nb_)]
                            row[f"msr_{model}"] = np.mean(res)
                            if model == "hybrid":
                                row["alpha"], row["q"] = a, q
                                row["avoid_rate_L1"], row["avoid_rate_L2"] = q, a * q
                        rows.append(row)
                    D = pd.DataFrame(rows)
                    long = D.melt(id_vars="mouse", value_vars=["msr_shock", "msr_avoid", "msr_hybrid"], var_name="model", value_name="msr")
                    aov = pg.rm_anova(long, dv="msr", within="model", subject="mouse", correction=True)
                    RW[(rd, form, st, trials, seed)] = D
                    means = D[["msr_shock", "msr_avoid", "msr_hybrid"]].mean()
                    pgg = aov["p_GG_corr"].iloc[0] if "p_GG_corr" in aov else aov["p_unc"].iloc[0]
                    # Hybrid against avoid-only, paired over mice: reported, decides nothing (the criterion uses the ANOVA).
                    p_ha = stats.ttest_rel(D.msr_hybrid, D.msr_avoid).pvalue
                    rw_rows.append(dict(tone=rd, update=form, start=st, trials=trials, seed=seed, mice=len(D), **{k[4:]: v for k, v in means.items()},
                                        F=aov["F"].iloc[0], p=aov["p_unc"].iloc[0], p_GG=pgg, hybrid_lowest=means.idxmin() == "msr_hybrid", p_hybrid_vs_avoid=p_ha,
                                        reproduced=bool(means.idxmin() == "msr_hybrid" and pgg < 0.05),
                                        alpha_range=f"{D.alpha.min():.2f} to {D.alpha.max():.2f}", q_range=f"{D.q.min():.2f} to {D.q.max():.2f}"))
RWT = pd.DataFrame(rw_rows)
print(RWT.round(4).to_string(index=False))
for form in ("U1", "U2"):
    print(f"\nPer mouse, clock, {form}, code start, T1 S1:")
    print(RW[("clock", form, "code", "T1", "S1")].round(4).to_string(index=False))

**Result: reproduced.** On the `clock` windows, under all four readings (T1 and T2, S1 and S2) the
hybrid model has the lowest mean squared residual over blocks (0.0048 to 0.0076, against 0.0243 to
0.0364 for shock-only and 0.0167 to 0.0221 for avoid-only), and the repeated measures ANOVA gives
Geisser-Greenhouse p = 0.012 to 0.050. The U1 fit (the claims file's update, R(t+1)) stays at alpha = 1
in every mouse from the code's start and from four others. With the Methods' formula (U2, R_t) the
fitted alpha ranges from 0.00 to 1.00 across mice (alpha = 0 in F2, a mouse with 9
successes in 9 day 1 trials, where all three models give the same residual), and the hybrid model is still lowest (0.0184 to
0.0231, avoid-only 0.0197 to 0.0263, shock-only 0.12 to 0.18; GG p = 0.0036 to 0.0070), only just below
avoid-only. The ANOVA does not compare hybrid with avoid-only; paired over mice on the `clock` windows
(reported, deciding nothing) the hybrid model is below avoid-only with p = 0.0035 to 0.0073 under U1 and p = 0.078 to 0.35
under U2. With the paper's tone bouts T2 holds (GG p = 0.020) and T1 does not (GG p = 0.097 and 0.11)
under U1; U2 holds under all four. Whether the update uses R(t+1) is asked of a reader
(questions/gabriel-2025.yaml).

## Behaviour in the CILDS epochs (Figure S3G, H)

In [ ]:
rows = []
for rd in TONE:
    T = TT[rd]
    for s in S:
        if s[0] != 1:
            continue
        b = beh[s]
        succ = T[(T.day == 1) & (T.group == s[1]) & (T.mouse == s[2])].set_index("trial").success
        first = next((k for k in range(4, 13) if succ.get(k) == 1), None)
        for ep, (t0, t1) in (("onset", (0, 5)), ("preshock", (23, 28))):
            for which in ("platform", "freezing"):
                v = pma.epoch_fraction(b, rd, t0, t1, which)
                rows.append(dict(tone=rd, group=s[1], mouse=s[2], epoch=ep, measure=which, B1=np.nanmean(v[3:12]),
                                 A2_first_avoid=v[first - 1] if first else np.nan,
                                 A1_after_first_avoid=v[first] if first and first < 12 else np.nan, first_avoid=first))
G = pd.DataFrame(rows)
res = []
for (rd, ep, meas), d in G.groupby(["tone", "epoch", "measure"]):
    for rg in ("B1", "A2_first_avoid", "A1_after_first_avoid"):
        a, b = d[d.group == "Shock"][rg].dropna(), d[d.group == "NonShock"][rg].dropna()
        t = stats.ttest_ind(a, b, equal_var=False)
        res.append(dict(tone=rd, epoch=ep, measure=meas, reading=rg, n_shock=len(a), n_ns=len(b), shock=a.mean(), nonshock=b.mean(), p=t.pvalue))
GR = pd.DataFrame(res)
print(GR.round(4).to_string(index=False))
for (rd, rg), d in GR.groupby(["tone", "reading"]):
    print(f"{rd} {rg}: comparisons with p < 0.05: {int((d.p < 0.05).sum())} of {len(d)}")
for rd in TONE:
    print(f"{rd} first avoidance trial per mouse:", G[G.tone == rd].drop_duplicates("mouse")[["group", "mouse", "first_avoid"]].values.tolist())

**Result: reproduced.** Time on the platform and freezing in the tone-onset (0 to 5 s) and pre-shock (23
to 28 s) epochs of day 1 do not differ between groups under any trial or tone reading (24 Welch tests,
smallest p = 0.11; for example platform at onset over trials 4 to 12, 0.63 against 0.42, p = 0.14).

In [ ]:
print(f"Notebook run time {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| zenodo-dandiset-link | partially reproduced |
| data-neurons | reproduced |
| data-registered-cells, fig1o-tone-to-mixed, fig1o-nonmod-to-mixed | blocked: not runnable (CellReg is MATLAB) |
| fig1e-platform-time, fig1f-successful-trials | reproduced |
| fig1g-freezing, fig1h-latency | blocked: methods underspecified (the Sidak families disagree) |
| fig1kl-overall-encoding-similar | blocked: methods underspecified |
| fig1kl-day1-categories-similar | not reproduced |
| disc-tone-cells-baseline | blocked: methods underspecified |
| fig1k-mixed-day2 | not reproduced |
| fig1p-platform-exit-excited-day2 | partially reproduced |
| fig1q-platform-exit-inhibited-day2, fig1pq-freezing-day2 | reproduced |
| fig1r-mixed-exit-day1 | partially reproduced (assumed: reading M2) |
| figs1h-exit-not-shock | reproduced |
| figs2-innate-behaviors | blocked: not runnable (proposed: not reproduced, the escape part fails) |
| fig2-day1-tone-stronger | reproduced |
| fig2-day2-tone-further | partially reproduced (assumed: reading H1) |
| fig2e-modulation-entries | not reproduced |
| fig2f-modulation-exits | reproduced |
| fig3a-decoding-shock-gt-ns | reproduced (assumed: C1 N1, `clock` tone windows; proposed: partially reproduced, the pre-shock part does not exceed the shuffled-label control) |
| fig3a-decoding-before-tone | reproduced (assumed: C1 N1, `clock` tone windows) |
| fig3a-decoding-peak-preshock | not reproduced |
| fig3b-feature-cells-enhance | not reproduced (assumed: C1 N1, `clock` tone windows) |
| fig3c-nonmod-cells-worsen | blocked: methods underspecified |
| fig4b-hybrid-model-best | reproduced (assumed: `clock` tone windows) |
| figs3gh-behavior-similar | reproduced |
| 13 CILDS claims (fig4e to fig4l, figs3ab, figs3cd, figs3ef, figs4ab, figs4cd) | blocked: not runnable |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's: every claim is tested on the Zenodo record, as the dandiset
holds no calcium imaging; the behaviour-only mice of Figure 1 are not deposited (9 and 7 mice on day 1,
7 and 7 on day 2); the tone windows are corrected to the camera clock in six sessions where the camera
ran near 30 frames/s; the ROC routine (caAnalyze), the alignment of calcium and behaviour frames, and
the cell sets of the decoding (ROC_IDs) are not public and were rebuilt from the Methods; the Sidak
comparisons use t-tests corrected for the family, not Prism's pooled error, while Fisher's LSD uses the
pooled residual mean square of the two-way ANOVA, as Prism does. Every ROC, decoding and model result is
shown on the paper's tone bouts too, the decoding cell sets of Figure 3B and 3C on the paper's bouts for
the P1-F1 reading only, and the mean-activity and Figure 2 results with four shuffle seeds, as several
sit near p = 0.05. The decoders of Figure 3A are compared with decoders trained on shuffled trial
outcomes, which shows that the shocked mice's pre-shock and pre-tone values are no higher than with
shuffled outcomes. The event-call counts stay below the paper's for day 1 tone onsets and day 2 shocked
platform exits under every event window tried; other counts reach or pass the paper's in some variants
(for example day 1 non-shocked platform entries, 111/79 against 95/66).

What was found about the dataset: dandiset 001887 holds no imaging, behaviour labels for 2 of 30
sessions, a trial table copied from another mouse (F7 day 2), shock trials in a non-shocked session (F3
day 1) and conversion dates as session start times (F1, F2). The Zenodo record holds the imaged sessions
with the paper's cell counts, the cell footprints and no cell registration; its BehaviorDEPOT output
holds no jumping, rearing or escape labels (escape can be computed from the platform labels, as
the Methods define it).